# Notebook 15 — Beaten-Distance Semantics

## Purpose

This notebook investigates the source fields `ovr_btn` and `btn` before either field is parsed, cleaned, renamed or used in the future database.

The bounded question is:

> What do `ovr_btn` and `btn` represent in the source, how are they related within a race, and which values can be interpreted or derived safely without inventing information?

The notebook does not assume that:

* `btn` always means the distance behind the immediately preceding finisher;
* `ovr_btn` always means the distance behind the winner;
* either field is always a cumulative version of the other;
* zero always identifies the winner;
* all numeric values apply only to finishers;
* non-finishers should always have null values; or
* arithmetic inconsistencies are necessarily source errors.

The investigation will eventually establish:

1. how `ovr_btn` and `btn` are stored;
2. their null, blank, numeric and textual value patterns;
3. how each field relates to raw finishing position;
4. how winners, ties, dead heats and non-finishers are represented;
5. whether either field describes a preceding-runner or winner-relative distance;
6. whether one field can be derived reliably from the other;
7. what rounding, incomplete and contradictory states exist;
8. whether meanings differ by jurisdiction, race type, year or course;
9. which raw and interpreted states should be preserved in the future database; and
10. which validation failures should block ingestion.

The immediate first stage is restricted to raw profiling:

* SQLite storage classes;
* null and blank counts;
* distinct raw values;
* numeric ranges; and
* relationships with raw `pos`.

No within-race arithmetic classification will be designed until that evidence has been reviewed.

## Grain and source

The source grain is one runner row.

Provisional races are identified using:

`date + course + off`

The supplied `race_id` remains source lineage and is not treated as a unique race key.

All raw-source queries must:

* open SQLite in read-only mode;
* query the `data` table;
* preserve `ovr_btn`, `btn` and `pos` unchanged; and
* exclude the header-like first row using `rowid <> 1`.

The raw source database is:

`data/raw/form_2015-present/form_2015-present/raceform.db`

The governed data-row predicate is:

```python
DATA_ROW_PREDICATE = "rowid <> 1"
```


In [1]:
from pathlib import Path
import sqlite3

import pandas as pd


def find_repository_root(start: Path) -> Path:
    """Return the nearest parent containing the repository's pyproject.toml."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError("Could not locate the repository root.")


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())

SOURCE_DATABASE = (
    REPOSITORY_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"

if not SOURCE_DATABASE.exists():
    raise FileNotFoundError(f"Source database not found: {SOURCE_DATABASE}")

SOURCE_DATABASE_URI = f"file:{SOURCE_DATABASE.as_posix()}?mode=ro"

with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    source_columns = pd.read_sql_query(
        f"PRAGMA table_info({SOURCE_TABLE})",
        connection,
    )

required_columns = {"date", "course", "off", "race_id", "pos", "ovr_btn", "btn"}
available_columns = set(source_columns["name"])
missing_columns = required_columns - available_columns

if missing_columns:
    raise ValueError(
        "Required source columns are missing: "
        + ", ".join(sorted(missing_columns))
    )

pd.DataFrame(
    {
        "check": [
            "repository root",
            "source database",
            "SQLite access",
            "source table",
            "governed predicate",
            "source columns",
            "required study columns",
        ],
        "result": [
            str(REPOSITORY_ROOT),
            str(SOURCE_DATABASE.relative_to(REPOSITORY_ROOT)),
            "read-only",
            SOURCE_TABLE,
            DATA_ROW_PREDICATE,
            len(source_columns),
            ", ".join(sorted(required_columns)),
        ],
    }
)

,check,result
0,repository root,/home/rob/Documents/inside-rails-horse-racing
1,source database,data/raw/form_2015-present/form_2015-present/r...
2,SQLite access,read-only
3,source table,data
4,governed predicate,rowid <> 1
5,source columns,37
6,required study columns,"btn, course, date, off, ovr_btn, pos, race_id"


## 1. Bounded raw source profile

**Table grain:** one row per studied source field.

This first profile covers only:

* SQLite storage classes;
* null and blank-text counts;
* distinct raw values;
* numeric minimum and maximum values;
* negative and zero values;
* non-numeric populated values; and
* availability against raw `pos`.

Raw values remain unchanged. Numeric summaries use SQLite numeric storage classes only and do not yet imply that every stored number has a valid beaten-distance meaning.


In [3]:
# Build a runner-row profile for each beaten-distance source field.
#
# Input grain:
#   One source runner row from the governed source population.
#
# Output grain:
#   One summary row per studied field: `ovr_btn` and `btn`.
#
# Important:
#   - Raw source values are not changed.
#   - Numeric summaries apply only to values physically stored by SQLite
#     as INTEGER or REAL.
#   - Text that might resemble a number is not parsed at this stage.
#   - No semantic meaning is assigned to zero, negative or populated values yet.

profile_query = f"""
WITH field_values AS (
    -- Reshape raw `ovr_btn` values into a common field/value structure.
    SELECT
        'ovr_btn' AS field,
        ovr_btn AS raw_value,
        typeof(ovr_btn) AS storage_class
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}

    UNION ALL

    -- Reshape raw `btn` values into the same structure.
    SELECT
        'btn' AS field,
        btn AS raw_value,
        typeof(btn) AS storage_class
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
)
SELECT
    field,

    -- Total governed runner rows examined for this field.
    COUNT(*) AS runner_rows,

    -- Raw missing-value states.
    SUM(raw_value IS NULL) AS null_rows,
    SUM(
        storage_class = 'text'
        AND TRIM(CAST(raw_value AS TEXT)) = ''
    ) AS blank_text_rows,

    -- Distinct raw values, excluding SQLite NULL automatically.
    COUNT(DISTINCT raw_value) AS distinct_nonnull_raw_values,

    -- Physical SQLite storage classes.
    SUM(storage_class = 'integer') AS integer_rows,
    SUM(storage_class = 'real') AS real_rows,
    SUM(storage_class = 'text') AS text_rows,
    SUM(storage_class = 'blob') AS blob_rows,

    -- Numeric range restricted to physically numeric SQLite values.
    MIN(
        CASE
            WHEN storage_class IN ('integer', 'real')
            THEN raw_value
        END
    ) AS minimum_stored_numeric_value,
    MAX(
        CASE
            WHEN storage_class IN ('integer', 'real')
            THEN raw_value
        END
    ) AS maximum_stored_numeric_value,

    -- Potentially important numeric states retained for later investigation.
    SUM(
        storage_class IN ('integer', 'real')
        AND raw_value < 0
    ) AS negative_stored_numeric_rows,
    SUM(
        storage_class IN ('integer', 'real')
        AND raw_value = 0
    ) AS zero_stored_numeric_rows,

    -- Populated text remains unparsed and requires separate inspection.
    SUM(
        storage_class = 'text'
        AND TRIM(CAST(raw_value AS TEXT)) <> ''
    ) AS populated_text_rows
FROM field_values
GROUP BY field
ORDER BY field
"""

# Execute through SQLite's read-only URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    raw_field_profile = pd.read_sql_query(profile_query, connection)

# Display the two-row field-level profile for review.
raw_field_profile

,field,runner_rows,null_rows,blank_text_rows,distinct_nonnull_raw_values,integer_rows,real_rows,text_rows,blob_rows,minimum_stored_numeric_value,maximum_stored_numeric_value,negative_stored_numeric_rows,zero_stored_numeric_rows,populated_text_rows
0,btn,1851285,0,0,245,661250,1096043,93992,0,0,186.0,0,192162,93992
1,ovr_btn,1851285,0,0,951,589977,1167316,93992,0,0,331.5,0,189408,93992


### Initial storage findings

**Grain of the preceding table:** one summary row per source field.

The source contains no SQLite nulls or blank-text states in either `ovr_btn` or `btn`.

Both fields contain:

* physically numeric values stored as SQLite integers or reals; and
* 93,992 populated text values.

The two fields have the same populated-text row count, but this does not yet prove that the text occurs on the same runner rows or contains the same raw values.

No negative physically numeric values were observed.

The observed numeric ranges are:

* `btn`: 0 to 186;
* `ovr_btn`: 0 to 331.5.

`ovr_btn` has substantially more distinct raw values than `btn`. This is compatible with several possible relationships, including cumulative versus incremental distances, but it is not evidence sufficient to establish one.

Zero remains unresolved. It may represent winners, ties, unavailable distances, non-finishers or multiple source conventions.


In [5]:
# Profile the joint raw states of finishing position, `ovr_btn` and `btn`.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One row per distinct combination of:
#       raw `pos`,
#       its SQLite storage class,
#       raw `ovr_btn`,
#       raw `btn`,
#       and the storage classes of both distance fields.
#
# This deliberately preserves all source values unchanged.
# It does not parse finishing positions or distances and does not yet classify
# winners, finishers, ties or non-finishers.

raw_pos_distance_query = f"""
SELECT
    pos AS raw_pos,
    typeof(pos) AS pos_storage_class,
    ovr_btn AS raw_ovr_btn,
    typeof(ovr_btn) AS ovr_btn_storage_class,
    btn AS raw_btn,
    typeof(btn) AS btn_storage_class,
    COUNT(*) AS runner_rows
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
GROUP BY
    pos,
    typeof(pos),
    ovr_btn,
    typeof(ovr_btn),
    btn,
    typeof(btn)
ORDER BY
    runner_rows DESC,
    CAST(pos AS TEXT),
    CAST(ovr_btn AS TEXT),
    CAST(btn AS TEXT)
"""

# Execute the raw-state query through the read-only SQLite connection.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    raw_pos_distance_states = pd.read_sql_query(
        raw_pos_distance_query,
        connection,
    )

# Display the most frequent raw combinations first.
raw_pos_distance_states.head(30)

,raw_pos,pos_storage_class,raw_ovr_btn,ovr_btn_storage_class,raw_btn,btn_storage_class,runner_rows
0,1,integer,0,integer,0,integer,188844
1,PU,text,-,text,-,text,65832
2,2,integer,0.5,real,0.5,real,18381
3,2,integer,0.3,real,0.3,real,16699
4,F,text,-,text,-,text,15681
5,2,integer,0.75,real,0.75,real,14812
6,2,integer,1.25,real,1.25,real,13260
7,2,integer,0.2,real,0.2,real,10797
8,2,integer,1,integer,1,integer,10345
9,2,integer,1.5,real,1.5,real,9906


### Initial relationship with raw finishing position

**Grain of the preceding table:** one row per distinct raw combination of `pos`, `ovr_btn`, `btn` and their SQLite storage classes.

The most frequent raw states show:

* `pos = 1` commonly accompanied by numeric zero in both distance fields;
* `pos = 2` commonly accompanied by equal values in `ovr_btn` and `btn`;
* later numeric positions where `ovr_btn` and `btn` differ; and
* common textual non-finishing outcomes accompanied by the text sentinel `-` in both fields.

For example, some third-place rows contain:

* `ovr_btn = 0.5`;
* `btn = 0.3`.

This is compatible with a cumulative-versus-incremental relationship, but a frequent example does not prove the field definitions or their stability across the source.

The dominant winner combination contains 188,844 runner rows, compared with 189,043 established provisional races. The difference must be investigated rather than interpreted immediately. Possible explanations include tied winners, unusual winner states, missing winners, void races or other source conventions.

The next step is to classify raw finishing-position states and count the associated distance states across the full runner-row population.


In [6]:
# Summarise both beaten-distance fields against broad raw finishing-position states.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One row per broad raw `pos` category.
#
# Raw values remain unchanged in the source query. The `pos_category` column is
# a deliberately limited analytical classification used only to organise the
# initial profile:
#   - positive numeric positions are separated into winner and later finisher;
#   - known textual outcomes remain individually visible;
#   - any unanticipated state is retained as `other_text_or_unclassified`.
#
# This cell does not yet interpret either distance field semantically.
# In particular, zero is not automatically treated as a valid winner marker,
# and the text sentinel `-` is not automatically converted to null.

pos_state_profile_query = f"""
WITH runner_states AS (
    SELECT
        pos AS raw_pos,
        typeof(pos) AS pos_storage_class,
        ovr_btn AS raw_ovr_btn,
        typeof(ovr_btn) AS ovr_btn_storage_class,
        btn AS raw_btn,
        typeof(btn) AS btn_storage_class,

        CASE
            WHEN typeof(pos) IN ('integer', 'real') AND pos = 1
                THEN 'numeric_winner'
            WHEN typeof(pos) IN ('integer', 'real') AND pos > 1
                THEN 'later_numeric_finisher'
            WHEN typeof(pos) IN ('integer', 'real') AND pos <= 0
                THEN 'nonpositive_numeric_position'
            WHEN typeof(pos) = 'text' AND TRIM(CAST(pos AS TEXT)) = ''
                THEN 'blank_text_position'
            WHEN CAST(pos AS TEXT) IN (
                'PU', 'F', 'UR', 'RO', 'BD', 'R', 'RR',
                'DSQ', 'VOID', 'SU', 'CO', 'L'
            )
                THEN CAST(pos AS TEXT)
            WHEN pos IS NULL
                THEN 'null_position'
            ELSE 'other_text_or_unclassified'
        END AS pos_category
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
)
SELECT
    pos_category,
    COUNT(*) AS runner_rows,

    -- Number of distinct raw position values represented by the category.
    COUNT(DISTINCT raw_pos) AS distinct_raw_pos_values,

    -- Physical states of `ovr_btn`.
    SUM(ovr_btn_storage_class IN ('integer', 'real')) AS ovr_btn_numeric_rows,
    SUM(ovr_btn_storage_class = 'text') AS ovr_btn_text_rows,
    SUM(
        ovr_btn_storage_class IN ('integer', 'real')
        AND raw_ovr_btn = 0
    ) AS ovr_btn_zero_rows,
    SUM(
        ovr_btn_storage_class IN ('integer', 'real')
        AND raw_ovr_btn > 0
    ) AS ovr_btn_positive_rows,
    SUM(
        ovr_btn_storage_class = 'text'
        AND raw_ovr_btn = '-'
    ) AS ovr_btn_hyphen_rows,

    -- Physical states of `btn`.
    SUM(btn_storage_class IN ('integer', 'real')) AS btn_numeric_rows,
    SUM(btn_storage_class = 'text') AS btn_text_rows,
    SUM(
        btn_storage_class IN ('integer', 'real')
        AND raw_btn = 0
    ) AS btn_zero_rows,
    SUM(
        btn_storage_class IN ('integer', 'real')
        AND raw_btn > 0
    ) AS btn_positive_rows,
    SUM(
        btn_storage_class = 'text'
        AND raw_btn = '-'
    ) AS btn_hyphen_rows,

    -- Raw equality is useful for profiling but does not establish meaning.
    SUM(
        typeof(raw_ovr_btn) = typeof(raw_btn)
        AND raw_ovr_btn = raw_btn
    ) AS exactly_equal_raw_rows
FROM runner_states
GROUP BY pos_category
ORDER BY runner_rows DESC, pos_category
"""

# Execute the runner-row profile through SQLite's read-only URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    pos_state_profile = pd.read_sql_query(
        pos_state_profile_query,
        connection,
    )

# Confirm that the categories partition the complete governed population.
if int(pos_state_profile["runner_rows"].sum()) != 1_851_285:
    raise ValueError(
        "Raw finishing-position categories do not partition "
        "the governed runner-row population."
    )

# Display one summary row per raw finishing-position category.
pos_state_profile

,pos_category,runner_rows,distinct_raw_pos_values,ovr_btn_numeric_rows,ovr_btn_text_rows,ovr_btn_zero_rows,ovr_btn_positive_rows,ovr_btn_hyphen_rows,btn_numeric_rows,btn_text_rows,btn_zero_rows,btn_positive_rows,btn_hyphen_rows,exactly_equal_raw_rows
0,later_numeric_finisher,1567322,33,1567322,0,371,1566951,0,1567322,0,3121,1564201,0,197179
1,numeric_winner,189344,1,189344,0,188844,500,0,189344,0,188845,499,0,189340
2,PU,65832,1,0,65832,0,0,65832,0,65832,0,0,65832,65832
3,F,15681,1,0,15681,0,0,15681,0,15681,0,0,15681,15681
4,UR,9527,1,0,9527,0,0,9527,0,9527,0,0,9527,9527
5,BD,1020,1,0,1020,0,0,1020,0,1020,0,0,1020,1020
6,RR,723,1,0,723,0,0,723,0,723,0,0,723,723
7,DSQ,619,1,619,0,185,434,0,619,0,188,431,0,251
8,RO,463,1,0,463,0,0,463,0,463,0,0,463,463
9,SU,371,1,0,371,0,0,371,0,371,0,0,371,371


### Finishing-position states requiring identification

**Grain of the preceding table:** one summary row per broad raw finishing-position category.

The broad categories partition all 1,851,285 governed runner rows.

The profile establishes that:

* common non-finishing outcomes generally use the text sentinel `-` in both distance fields;
* disqualified runners retain numeric distance values and therefore require separate treatment;
* positive numeric winners are not represented uniformly;
* some later numeric finishers contain zero values;
* duplicate or tied positive finishing positions are likely because the number of `pos = 1` rows exceeds the number of provisional races; and
* two textual finishing-position values remain hidden inside the provisional `other_text_or_unclassified` category.

No conclusion is yet drawn about whether the exceptional numeric states are valid race-context conventions, tied positions, source ordering effects or source defects.

The next query identifies every raw finishing-position value and its associated distance-storage state before the analytical categories are revised.


In [7]:
# Identify every distinct raw finishing-position value in the source.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One summary row per distinct raw `pos` value and SQLite storage class.
#
# Purpose:
#   The previous profile grouped some raw position values into broad analytical
#   categories. This query exposes the original values so that no source state
#   remains concealed by an early classification.
#
# Important:
#   - `pos` remains raw and unparsed.
#   - `ovr_btn` and `btn` remain raw and unparsed.
#   - Equality counts describe source representation only; they do not establish
#     semantic equivalence between the two fields.

raw_position_value_query = f"""
SELECT
    pos AS raw_pos,
    typeof(pos) AS pos_storage_class,
    COUNT(*) AS runner_rows,

    -- Physical storage states for `ovr_btn`.
    SUM(typeof(ovr_btn) IN ('integer', 'real')) AS ovr_btn_numeric_rows,
    SUM(typeof(ovr_btn) = 'text') AS ovr_btn_text_rows,
    SUM(
        typeof(ovr_btn) IN ('integer', 'real')
        AND ovr_btn = 0
    ) AS ovr_btn_zero_rows,
    SUM(
        typeof(ovr_btn) IN ('integer', 'real')
        AND ovr_btn > 0
    ) AS ovr_btn_positive_rows,
    SUM(
        typeof(ovr_btn) = 'text'
        AND ovr_btn = '-'
    ) AS ovr_btn_hyphen_rows,

    -- Physical storage states for `btn`.
    SUM(typeof(btn) IN ('integer', 'real')) AS btn_numeric_rows,
    SUM(typeof(btn) = 'text') AS btn_text_rows,
    SUM(
        typeof(btn) IN ('integer', 'real')
        AND btn = 0
    ) AS btn_zero_rows,
    SUM(
        typeof(btn) IN ('integer', 'real')
        AND btn > 0
    ) AS btn_positive_rows,
    SUM(
        typeof(btn) = 'text'
        AND btn = '-'
    ) AS btn_hyphen_rows,

    -- Exact raw equality, including matching numeric values and text sentinels.
    SUM(
        typeof(ovr_btn) = typeof(btn)
        AND ovr_btn = btn
    ) AS exactly_equal_raw_rows
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
GROUP BY
    pos,
    typeof(pos)
ORDER BY
    CASE
        WHEN typeof(pos) IN ('integer', 'real') THEN 0
        ELSE 1
    END,
    CASE
        WHEN typeof(pos) IN ('integer', 'real') THEN CAST(pos AS REAL)
    END,
    CAST(pos AS TEXT)
"""

# Execute through the read-only SQLite connection.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    raw_position_value_profile = pd.read_sql_query(
        raw_position_value_query,
        connection,
    )

# Confirm that the distinct raw-position groups still cover every governed row.
if int(raw_position_value_profile["runner_rows"].sum()) != 1_851_285:
    raise ValueError(
        "Distinct raw finishing-position values do not cover "
        "the governed runner-row population."
    )

# Display every raw position value; the result should be small enough for review.
raw_position_value_profile

,raw_pos,pos_storage_class,runner_rows,ovr_btn_numeric_rows,ovr_btn_text_rows,ovr_btn_zero_rows,ovr_btn_positive_rows,ovr_btn_hyphen_rows,btn_numeric_rows,btn_text_rows,btn_zero_rows,btn_positive_rows,btn_hyphen_rows,exactly_equal_raw_rows
0,0,integer,8,8,0,8,0,0,8,0,8,0,0,8
1,1,integer,189344,189344,0,188844,500,0,189344,0,188845,499,0,189340
2,2,integer,189007,189007,0,293,188714,0,189007,0,656,188351,0,188251
3,3,integer,188141,188141,0,34,188107,0,188141,0,466,187675,0,8725
4,4,integer,184662,184662,0,11,184651,0,184662,0,470,184192,0,144
5,5,integer,175886,175886,0,6,175880,0,175886,0,418,175468,0,15
6,6,integer,161327,161327,0,8,161319,0,161327,0,307,161020,0,17
7,7,integer,142993,142993,0,4,142989,0,142993,0,235,142758,0,6
8,8,integer,122268,122268,0,3,122265,0,122268,0,171,122097,0,5
9,9,integer,101726,101726,0,2,101724,0,101726,0,125,101601,0,2


### Complete raw position inventory

**Grain of the preceding table:** one summary row per distinct raw finishing-position value and SQLite storage class.

The source contains numeric finishing positions from `0` through `34`.

The previously unclassified textual values are:

* `REF`, observed on 291 runner rows;
* `LFT`, observed on 7 runner rows.

Both are accompanied by the text sentinel `-` in `ovr_btn` and `btn`.

All observed ordinary non-finishing outcomes use text values in both distance fields. `DSQ` is materially different because its 619 rows retain physically numeric values.

The observed textual finishing outcomes are therefore:

* `BD`;
* `CO`;
* `F`;
* `LFT`;
* `PU`;
* `REF`;
* `RO`;
* `RR`;
* `SU`;
* `UR`; and
* `DSQ`, which must remain analytically separate because its distance representation is numeric.

The eight `pos = 0` rows contain numeric zero in both fields. Their meaning remains unresolved pending race-context inspection.

This completes the inventory of raw `pos` values, but not yet the inventory of raw distance text states.


In [8]:
# Inspect every populated text value used by `ovr_btn` and `btn`.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One summary row per field, raw text value and associated raw `pos`.
#
# Purpose:
#   The earlier profiles showed 93,992 text rows in each distance field.
#   This query verifies the actual text sentinels rather than assuming that
#   every text value is a hyphen.
#
# Important:
#   - Raw field values are preserved exactly.
#   - No text value is converted to null or assigned a semantic meaning.
#   - Raw `pos` remains visible so that any alternative text convention can
#     be tied back to its finishing outcome.

distance_text_value_query = f"""
WITH text_values AS (
    -- Raw text states from `ovr_btn`.
    SELECT
        'ovr_btn' AS field,
        ovr_btn AS raw_distance_value,
        pos AS raw_pos,
        typeof(pos) AS pos_storage_class
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(ovr_btn) = 'text'

    UNION ALL

    -- Raw text states from `btn`.
    SELECT
        'btn' AS field,
        btn AS raw_distance_value,
        pos AS raw_pos,
        typeof(pos) AS pos_storage_class
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(btn) = 'text'
)
SELECT
    field,
    raw_distance_value,
    raw_pos,
    pos_storage_class,
    COUNT(*) AS runner_rows
FROM text_values
GROUP BY
    field,
    raw_distance_value,
    raw_pos,
    pos_storage_class
ORDER BY
    field,
    runner_rows DESC,
    CAST(raw_pos AS TEXT),
    raw_distance_value
"""

# Execute through SQLite's read-only URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    distance_text_value_profile = pd.read_sql_query(
        distance_text_value_query,
        connection,
    )

# Validate that the output accounts for every text row found earlier:
# 93,992 rows for each field, or 187,984 field-value observations in total.
expected_text_observations = 93_992 * 2
observed_text_observations = int(
    distance_text_value_profile["runner_rows"].sum()
)

if observed_text_observations != expected_text_observations:
    raise ValueError(
        "Text-value profile does not account for every observed "
        "distance-field text value."
    )

# Display the complete text-value inventory.
distance_text_value_profile

,field,raw_distance_value,raw_pos,pos_storage_class,runner_rows
0,btn,-,PU,text,65832
1,btn,-,F,text,15681
2,btn,-,UR,text,9527
3,btn,-,BD,text,1020
4,btn,-,RR,text,723
5,btn,-,RO,text,463
6,btn,-,SU,text,371
7,btn,-,REF,text,291
8,btn,-,CO,text,77
9,btn,-,LFT,text,7


## Initial bounded-profile review

### Scope completed

The first-stage raw profile covers:

* SQLite storage classes;
* null and blank-text states;
* distinct raw values;
* physically numeric ranges;
* negative and zero values;
* text-value inventory; and
* relationships with raw `pos`.

No within-race arithmetic or semantic parser has yet been designed.

### Runner-row findings

Both `ovr_btn` and `btn` are populated on all 1,851,285 governed runner rows.

Neither field contains:

* SQLite nulls;
* blank text;
* blobs; or
* negative physically numeric values.

Each field contains exactly 93,992 text rows. Every observed text value is the single raw sentinel:

`-`

The sentinel occurs only with these raw finishing outcomes:

* `PU`;
* `F`;
* `UR`;
* `BD`;
* `RR`;
* `RO`;
* `SU`;
* `REF`;
* `CO`; and
* `LFT`.

The two fields use the same `-` sentinel on the same categories of ordinary non-finishing outcomes.

`DSQ` is different. All 619 disqualified runner rows retain physically numeric values in both fields.

### Numeric ranges and diversity

The observed physically numeric ranges are:

* `btn`: 0 to 186;
* `ovr_btn`: 0 to 331.5.

The fields contain:

* 245 distinct non-null raw `btn` values;
* 951 distinct non-null raw `ovr_btn` values.

The greater range and value diversity of `ovr_btn` are compatible with a cumulative or winner-relative measure, but do not establish that meaning.

### Relationship with raw finishing position

The dominant winner representation is:

* `pos = 1`;
* `ovr_btn = 0`;
* `btn = 0`.

However, winner representation is not completely uniform:

* 500 `pos = 1` rows have positive `ovr_btn`;
* 499 `pos = 1` rows have positive `btn`;
* four winner rows have unequal raw distance values.

There are 189,344 `pos = 1` runner rows but only 189,043 provisional races. This proves that at least some races contain duplicate or tied first-place positions, although the exact race contexts have not yet been inspected.

Later numeric finishers are usually positive in both fields, but exceptions exist:

* 371 later finishers have zero `ovr_btn`;
* 3,121 later finishers have zero `btn`.

The two fields are usually equal for second-place runners. From third place onward, they commonly differ.

A frequent third-place pattern is compatible with:

* `ovr_btn` representing a distance from the winner; and
* `btn` representing a distance from the preceding finisher.

That remains a hypothesis rather than a conclusion until within-race ordering and arithmetic are tested.

### Raw finishing-position inventory

Raw numeric `pos` values run from 0 through 34.

Eight rows have:

* `pos = 0`;
* `ovr_btn = 0`;
* `btn = 0`.

Their meaning remains unresolved.

The complete observed textual outcomes are:

* `BD`;
* `CO`;
* `DSQ`;
* `F`;
* `LFT`;
* `PU`;
* `REF`;
* `RO`;
* `RR`;
* `SU`; and
* `UR`.

### Provisional hypotheses

The initial evidence supports testing, but does not yet prove, the following hypotheses:

1. `ovr_btn` is ordinarily a winner-relative distance for numeric finishers.
2. `btn` is ordinarily a margin from the immediately preceding governed finisher.
3. Second-place values are therefore ordinarily equal across the two fields.
4. The `-` sentinel represents distance not supplied or not applicable for ordinary non-finishers.
5. `DSQ` runners may retain a distance associated with an original or governed finishing position.
6. Zero among later finishers may represent ties, dead heats, source ordering conventions or exceptional source states.
7. Positive values on `pos = 1` rows may be associated with duplicate winners, dead heats, amended results or source inconsistencies.

None of these hypotheses should yet be implemented as a parser, cleaning rule or database constraint.

### Rejected conclusions at this stage

The evidence does not justify concluding that:

* every winner has zero in both fields;
* every beaten runner has a positive value;
* every non-finisher has `-`;
* `DSQ` means distance is unavailable;
* either field can be derived reliably from the other;
* unequal or unexpected values are source errors; or
* row order is a valid finishing order.

### Next bounded investigation

The next stage should inspect race context before attempting source-wide arithmetic classification.

It should begin with small lineage-preserving exception tables for:

* positive distance values on `pos = 1` rows;
* zero distances on later numeric finishers;
* duplicate positive finishing positions within provisional races;
* the eight `pos = 0` rows; and
* numeric distance values on `DSQ` rows.

Only after those examples are understood should cumulative arithmetic between `ovr_btn` and `btn` be designed.


## 2. Positive distance values on numeric winner rows

**Input grain:** one governed source runner row.

**Review-table grain:** one runner row where raw `pos = 1` and at least one distance field is positive.

This inspection preserves race and runner lineage so that positive winner distances can be reviewed in context.

It does not assume that these rows are errors. Possible explanations include:

* dead heats or tied first-place positions;
* amended or disqualified results;
* jurisdiction-specific conventions;
* source ordering effects;
* inconsistent winner-state encoding; or
* genuine source defects.

The first step is to inspect the raw rows and count the distinct provisional races involved.


In [10]:
# Extract every numeric winner row with a positive value in either distance field.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One runner row satisfying:
#       raw `pos = 1`
#       and (`ovr_btn > 0` or `btn > 0`).
#
# Purpose:
#   The bounded profile found 500 winner rows with positive `ovr_btn` and
#   499 with positive `btn`. This cell preserves enough race and runner context
#   to determine whether those states are associated with ties, duplicate
#   winners, disqualifications, unusual jurisdictions or other conventions.
#
# Important:
#   - Raw `pos`, `ovr_btn` and `btn` remain unchanged.
#   - The candidate race identity remains `date + course + off`.
#   - `race_id` and SQLite `rowid` are retained as source lineage only.
#   - No row is labelled erroneous at this stage.

positive_winner_query = f"""
SELECT
    rowid AS source_rowid,
    date,
    course,
    off,
    race_id,
    race_name,
    type,
    horse,
    pos AS raw_pos,
    ovr_btn AS raw_ovr_btn,
    typeof(ovr_btn) AS ovr_btn_storage_class,
    btn AS raw_btn,
    typeof(btn) AS btn_storage_class
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND typeof(pos) IN ('integer', 'real')
  AND pos = 1
  AND (
      (
          typeof(ovr_btn) IN ('integer', 'real')
          AND ovr_btn > 0
      )
      OR
      (
          typeof(btn) IN ('integer', 'real')
          AND btn > 0
      )
  )
ORDER BY
    date,
    course,
    off,
    source_rowid
"""

# Execute through SQLite's read-only URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    positive_winner_rows = pd.read_sql_query(
        positive_winner_query,
        connection,
    )

# Validate the runner-row count against the earlier bounded profile.
# There should be 500 rows because every positive-`btn` winner is expected
# to be contained within the 500 positive-`ovr_btn` winner rows, but the
# assertion is intentionally based on the observed source result.
if len(positive_winner_rows) != 500:
    raise ValueError(
        "Positive winner-row count differs from the bounded profile: "
        f"{len(positive_winner_rows):,}"
    )

# Count the distinct provisional races represented by these runner rows.
positive_winner_race_count = (
    positive_winner_rows[["date", "course", "off"]]
    .drop_duplicates()
    .shape[0]
)

pd.DataFrame(
    {
        "measure": [
            "positive winner runner rows",
            "distinct provisional races",
            "minimum raw ovr_btn",
            "maximum raw ovr_btn",
            "minimum raw btn",
            "maximum raw btn",
            "rows where raw fields differ",
        ],
        "value": [
            len(positive_winner_rows),
            positive_winner_race_count,
            positive_winner_rows["raw_ovr_btn"].min(),
            positive_winner_rows["raw_ovr_btn"].max(),
            positive_winner_rows["raw_btn"].min(),
            positive_winner_rows["raw_btn"].max(),
            int(
                (
                    positive_winner_rows["raw_ovr_btn"]
                    != positive_winner_rows["raw_btn"]
                ).sum()
            ),
        ],
    }
)

,measure,value
0,positive winner runner rows,500.00
1,distinct provisional races,499.00
2,minimum raw ovr_btn,0.05
3,maximum raw ovr_btn,30.25
4,minimum raw btn,0.00
5,maximum raw btn,26.00
6,rows where raw fields differ,4.00


### Positive winner exceptions requiring immediate inspection

**Grain of the preceding table:** one summary row per measured property of the positive-winner subset.

The 500 positive winner rows occur in 499 provisional races. Therefore, one provisional race contains two positive-distance rows with raw `pos = 1`.

Only four positive winner rows contain unequal raw values between `ovr_btn` and `btn`.

These small exception sets should be inspected before drawing conclusions from the much larger group where the two fields are equal.

The next review tables preserve complete race context for:

* the provisional race containing multiple positive winner rows; and
* the four winner rows where `ovr_btn` and `btn` differ.


In [11]:
# Identify the provisional race containing more than one positive winner row
# and the four positive winner rows where the distance fields differ.
#
# Input grain:
#   `positive_winner_rows` contains one positive-distance winner runner row.
#
# Output grains:
#   `positive_winner_duplicate_races`:
#       one summary row per provisional race containing multiple positive
#       winner rows.
#
#   `positive_winner_unequal_rows`:
#       one runner row where raw `pos = 1` and raw `ovr_btn != btn`.
#
# Purpose:
#   These are the smallest and most informative exception groups within the
#   positive-winner population. They may reveal ties, duplicated positions,
#   unusual ordering or distinct source conventions.
#
# Important:
#   - Raw values remain unchanged.
#   - Candidate race identity is still `date + course + off`.
#   - No exception is labelled as an error.

positive_winner_duplicate_races = (
    positive_winner_rows
    .groupby(
        ["date", "course", "off"],
        dropna=False,
        as_index=False,
    )
    .agg(
        positive_winner_rows=("source_rowid", "size"),
        distinct_horses=("horse", "nunique"),
        minimum_raw_ovr_btn=("raw_ovr_btn", "min"),
        maximum_raw_ovr_btn=("raw_ovr_btn", "max"),
        minimum_raw_btn=("raw_btn", "min"),
        maximum_raw_btn=("raw_btn", "max"),
    )
    .query("positive_winner_rows > 1")
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

positive_winner_unequal_rows = (
    positive_winner_rows.loc[
        positive_winner_rows["raw_ovr_btn"]
        != positive_winner_rows["raw_btn"]
    ]
    .sort_values(
        ["date", "course", "off", "source_rowid"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Validate both exception counts against the earlier summary.
if len(positive_winner_duplicate_races) != 1:
    raise ValueError(
        "Expected exactly one provisional race with multiple positive "
        "winner rows."
    )

if len(positive_winner_unequal_rows) != 4:
    raise ValueError(
        "Expected exactly four positive winner rows with unequal "
        "distance fields."
    )

print("Provisional race with multiple positive winner rows:")
display(positive_winner_duplicate_races)

print("\nPositive winner rows where raw distance fields differ:")
display(
    positive_winner_unequal_rows[
        [
            "source_rowid",
            "date",
            "course",
            "off",
            "race_id",
            "race_name",
            "type",
            "horse",
            "raw_pos",
            "raw_ovr_btn",
            "raw_btn",
        ]
    ]
)

Provisional race with multiple positive winner rows:


,date,course,off,positive_winner_rows,distinct_horses,minimum_raw_ovr_btn,maximum_raw_ovr_btn,minimum_raw_btn,maximum_raw_btn
0,2024-06-29,San Isidro (ARG),10:40,2,2,2.0,2.0,0.0,2.0



Positive winner rows where raw distance fields differ:


,source_rowid,date,course,off,race_id,race_name,type,horse,raw_pos,raw_ovr_btn,raw_btn
0,178312,2016-03-12,Gavea (BRZ),7:35,646248,Longines Gran Premio Latinoamericano (3yo+) (...,Flat,Some In Tieme (BRZ),1,0.75,0.3
1,698485,2019-04-20,Gavea (BRZ),8:39,728424,Grande Premio Zelia Gonzaga Peixoto de Castro ...,Flat,Naomi Broadway (BRZ),1,1.50,1.0
2,928683,2020-11-15,Fontwell,3:50,769370,From The Horses Mouth Podcast Mares Handicap H...,Hurdle,Dharma Rain (IRE),1,30.25,0.5
3,1530126,2024-06-29,San Isidro (ARG),10:40,871811,Gran Premio Estrellas Mile (3yo+) (Round) (Turf),Flat,Folie Ninja (ARG),1,2.00,0.0


### Full race context for unequal positive winner rows

The four unequal positive winner rows occur in four provisional races:

* two at Gavea in Brazil;
* one at Fontwell in Great Britain;
* one at San Isidro in Argentina.

The Fontwell value is particularly unusual because the raw winner row contains:

* `ovr_btn = 30.25`;
* `btn = 0.5`.

The San Isidro race is also the only provisional race containing two positive-distance rows with raw `pos = 1`.

The next table retrieves every source runner row from these four races. It preserves:

* physical source row order;
* candidate race identity;
* horse identity;
* raw finishing position;
* raw distance values; and
* selected result context.

No arithmetic or finishing-order correction is applied.


In [13]:
# Retrieve every runner row from the four provisional races containing
# unequal positive winner-distance values.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One source runner row belonging to one of the four selected provisional
#   races, identified by `date + course + off`.
#
# Purpose:
#   A winner row cannot be interpreted safely in isolation. This review shows
#   all runners in each affected race so that duplicate finishing positions,
#   row ordering, disqualifications, ties and neighbouring margins can be seen.
#
# Important:
#   - `source_rowid` preserves physical SQLite row order as lineage only.
#   - Raw `pos`, `ovr_btn` and `btn` remain unchanged.
#   - No assumption is made that physical row order equals finishing order.
#   - No governed arithmetic classification is created in this cell.

# Build the four-race key set directly from the observed unequal winner rows.
unequal_winner_race_keys = (
    positive_winner_unequal_rows[
        ["date", "course", "off"]
    ]
    .drop_duplicates()
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Create a parameterised SQL condition rather than interpolating source values.
race_key_condition = " OR ".join(
    ["(date = ? AND course = ? AND off = ?)"]
    * len(unequal_winner_race_keys)
)

race_key_parameters = [
    value
    for race_key in unequal_winner_race_keys.itertuples(index=False, name=None)
    for value in race_key
]

unequal_winner_race_context_query = f"""
SELECT
    rowid AS source_rowid,
    date,
    course,
    off,
    race_id,
    race_name,
    type,
    horse,
    pos AS raw_pos,
    typeof(pos) AS pos_storage_class,
    ovr_btn AS raw_ovr_btn,
    typeof(ovr_btn) AS ovr_btn_storage_class,
    btn AS raw_btn,
    typeof(btn) AS btn_storage_class,
    ran AS raw_ran,
    num AS raw_num,
    comment AS raw_comment
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND ({race_key_condition})
ORDER BY
    date,
    course,
    off,
    source_rowid
"""

# Execute through SQLite's read-only URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    unequal_winner_race_context = pd.read_sql_query(
        unequal_winner_race_context_query,
        connection,
        params=race_key_parameters,
    )

# Confirm that all four selected provisional races were returned.
returned_race_count = (
    unequal_winner_race_context[
        ["date", "course", "off"]
    ]
    .drop_duplicates()
    .shape[0]
)

if returned_race_count != 4:
    raise ValueError(
        "Expected context for four provisional races, "
        f"but found {returned_race_count}."
    )

# Display each race separately so physical row order and duplicate positions
# remain easy to inspect.
for race_key, race_rows in unequal_winner_race_context.groupby(
    ["date", "course", "off"],
    sort=False,
    dropna=False,
):
    race_date, race_course, race_off = race_key

    print(f"{race_date} — {race_course} — {race_off}")

    display(
        race_rows[
            [
                "source_rowid",
                "horse",
                "raw_pos",
                "raw_ovr_btn",
                "raw_btn",
                "raw_ran",
                "raw_num",
                "raw_comment",
            ]
        ].reset_index(drop=True)
    )

2016-03-12 — Gavea (BRZ) — 7:35


,source_rowid,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,178277,Energia Garoa (BRZ),11,8.25,0.75,16,3,
1,178278,Sensacionale (USA),12,13,4.75,16,8,
2,178279,Bartholomeu (BRZ),13,13.75,0.75,16,16,
3,178280,Quiz Kid (ARG),14,14,0.3,16,9,
4,178281,Harlans Blue (ARG),15,14.75,0.75,16,5,
5,178282,Street Lolo (ARG),16,16.25,1.5,16,1,
6,178294,Incentive Boy (ARG),10,7.5,2.25,16,15,
7,178295,Nieto Mireyo (PER),9,5.25,0.3,16,14,
8,178296,Paint Naif (BRZ),8,5,1.5,16,10,
9,178297,Flyer (CHI),5,3.5,0.3,16,4,Finished 7th - later placed 5th


2019-04-20 — Gavea (BRZ) — 8:39


,source_rowid,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,698485,Naomi Broadway (BRZ),1,1.5,1,8,4,Finished 3rd - awarded the race
1,698494,Gaivina (BRZ),2,0,0,8,5,Finished 1st - disqualified and placed 2nd
2,698496,Grandeza (BRZ),3,0.5,0.5,8,1,Finished 2nd - disqualified and placed 3rd
3,698497,Kassies Angel (BRZ),5,3,1.25,8,7,
4,698498,Perigoosa (BRZ),6,4.25,1.25,8,2,
5,698499,Jurere Girl (BRZ),7,14.5,10.25,8,6,
6,698500,Lisboeta (BRZ),8,18,3.5,8,3,
7,698505,Midsummer Rain (BRZ),4,1.75,0.2,8,8,


2020-11-15 — Fontwell — 3:50


,source_rowid,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,928677,Just Henny (GB),DSQ,0,0,9,7,In touch with leaders - went third after 7th -...
1,928678,Havacuppa (GB),DSQ,2.25,2.25,9,6,Tracked leaders - prominent 7th - took wrong r...
2,928679,Kalaya (IRE),DSQ,2.25,0.1,9,4,Prominent - led 5th - took wrong route 3 out -...
3,928680,Kentford Mallard (GB),DSQ,6.75,4.5,9,1,Held up in rear - headway after 6th - went fou...
4,928681,Queen Among Kings (IRE),DSQ,22.75,16,9,5,Held up in rear - headway 6th - took wrong rou...
5,928682,Lucky Circle (GB),DSQ,29.75,7,9,8,Took keen hold - in touch with leaders - took ...
6,928683,Dharma Rain (IRE),1,30.25,0.5,9,3,Held up in rear - pushed along and raced in la...
7,928684,Remember Me Well (IRE),DSQ,59.25,29,9,10,Led - prominent when mistake and headed 5th - ...
8,928685,Bluebell Sally (IRE),PU,-,-,9,9,Tracked leaders - pulled up after 4th - lost a...


2024-06-29 — San Isidro (ARG) — 10:40


,source_rowid,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1530090,Puro De Cuba (ARG),14,27.25,3.5,16,6,
1,1530091,Sell Side (ARG),15,28.25,1,16,8,
2,1530092,El Exito (ARG),16,29,0.75,16,14,
3,1530109,El Patio (ARG),13,23.75,1,16,12,
4,1530110,Love The Races (ARG),12,22.75,0.3,16,16,
5,1530111,Negador Serial (ARG),11,22.5,1,16,13,
6,1530112,Forty Fain (ARG),10,21.5,9,16,1,
7,1530124,El Eminente (ARG),3,0,0,16,9,Finished 1st - disqualified and placed 3rd
8,1530125,Bronx (ARG),1,2,2,16,3,Finished dead-heat 2nd - placed dead-heat 1st
9,1530126,Folie Ninja (ARG),1,2,0,16,7,Finished dead-heat 2nd - placed dead-heat 1st


### Amended results preserve original finishing distances

**Grain of the preceding review tables:** one source runner row within each of four selected provisional races.

The four winner rows with unequal `ovr_btn` and `btn` are explained by amended race results.

In each case, the raw finishing position reflects a later official placing, while the distance values appear to retain the runner’s original physical position in the finishing sequence.

Observed examples include:

* a horse originally finishing third and later being awarded first place;
* a horse becoming the winner after runners ahead were disqualified;
* a horse awarded first despite finishing more than 30 lengths behind the original leader; and
* two runners promoted from a dead heat for second into a dead heat for first.

The San Isidro example also provides direct evidence for the likely relationship between the fields:

* both promoted runners have `ovr_btn = 2`, consistent with being two lengths behind the original winner;
* the first promoted runner has `btn = 2`;
* the second promoted runner has `btn = 0`, consistent with a dead heat with the immediately preceding runner.

This supports, but does not yet establish source-wide, the hypotheses that:

* `ovr_btn` ordinarily describes the runner’s distance behind the original physical winner;
* `btn` ordinarily describes the margin from the immediately preceding runner in the original physical finishing sequence; and
* amended official positions do not necessarily cause the distance fields to be recomputed.

Consequently, arithmetic based only on the current raw `pos` order would produce false contradictions in amended-result races.

Future within-race analysis must distinguish:

* governed official finishing position;
* inferred original physical finishing sequence;
* amended-result status; and
* distance arithmetic consistency.

Positive distance values on official winner rows must not be treated automatically as ingestion failures.


In [16]:
# Reload every positive-distance winner row with the source comment included.
#
# Input grain:
#   One governed source runner row.
#
# Intermediate output grain:
#   One runner row where:
#       raw `pos = 1`
#       and (`ovr_btn > 0` or `btn > 0`).
#
# Final output grain:
#   One summary row per exploratory comment-evidence state.
#
# Purpose:
#   The four unequal winner rows were explained by amended-result comments.
#   This cell tests how much of the complete positive-winner population contains
#   similar explicit source evidence.
#
# Important:
#   - Raw `pos`, `ovr_btn`, `btn` and `comment` are preserved unchanged.
#   - The keyword classification is exploratory notebook logic only.
#   - A blank comment does not prove that no amendment occurred.
#   - No result is labelled erroneous.

positive_winner_comment_query = f"""
SELECT
    rowid AS source_rowid,
    date,
    course,
    off,
    race_id,
    race_name,
    type,
    horse,
    pos AS raw_pos,
    ovr_btn AS raw_ovr_btn,
    btn AS raw_btn,
    comment AS raw_comment
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND typeof(pos) IN ('integer', 'real')
  AND pos = 1
  AND (
      (
          typeof(ovr_btn) IN ('integer', 'real')
          AND ovr_btn > 0
      )
      OR
      (
          typeof(btn) IN ('integer', 'real')
          AND btn > 0
      )
  )
ORDER BY
    date,
    course,
    off,
    source_rowid
"""

# Execute through SQLite's read-only URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    positive_winner_comment_profile = pd.read_sql_query(
        positive_winner_comment_query,
        connection,
    )

# Confirm that the reloaded population matches the earlier 500-row result.
if len(positive_winner_comment_profile) != 500:
    raise ValueError(
        "Positive winner-row count differs from the earlier profile: "
        f"{len(positive_winner_comment_profile):,}"
    )

# Create a separate search-only version of the comment.
# The original source comment remains unchanged in `raw_comment`.
positive_winner_comment_profile["comment_search_text"] = (
    positive_winner_comment_profile["raw_comment"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Begin with the broadest residual state.
positive_winner_comment_profile["comment_evidence_state"] = (
    "no_explicit_amendment_evidence"
)

# Keep blank comments visible as a separate source state.
positive_winner_comment_profile.loc[
    positive_winner_comment_profile["comment_search_text"].eq(""),
    "comment_evidence_state",
] = "blank_comment"

# Identify explicit dead-heat wording.
positive_winner_comment_profile.loc[
    positive_winner_comment_profile["comment_search_text"].str.contains(
        r"dead[- ]?heat",
        regex=True,
        na=False,
    ),
    "comment_evidence_state",
] = "explicit_dead_heat"

# Identify explicit wording associated with an amended official result.
# This assignment follows the dead-heat assignment deliberately: comments
# containing both dead-heat and amendment terms are classified principally
# as amended results for this summary.
positive_winner_comment_profile.loc[
    positive_winner_comment_profile["comment_search_text"].str.contains(
        r"disqual|awarded|placed|promoted",
        regex=True,
        na=False,
    ),
    "comment_evidence_state",
] = "explicit_amended_result"

# Add a row-level flag for unequal raw distance values.
positive_winner_comment_profile["unequal_distance_fields"] = (
    positive_winner_comment_profile["raw_ovr_btn"]
    != positive_winner_comment_profile["raw_btn"]
)

# Create a stable provisional-race key for aggregation only.
positive_winner_comment_profile["provisional_race_key"] = list(
    zip(
        positive_winner_comment_profile["date"],
        positive_winner_comment_profile["course"],
        positive_winner_comment_profile["off"],
    )
)

# Summarise the exploratory comment states.
positive_winner_comment_summary = (
    positive_winner_comment_profile
    .groupby(
        "comment_evidence_state",
        as_index=False,
        dropna=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=("provisional_race_key", "nunique"),
        unequal_distance_rows=("unequal_distance_fields", "sum"),
        minimum_raw_ovr_btn=("raw_ovr_btn", "min"),
        maximum_raw_ovr_btn=("raw_ovr_btn", "max"),
    )
    .sort_values(
        ["runner_rows", "comment_evidence_state"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Confirm that every positive winner row is represented exactly once.
if int(positive_winner_comment_summary["runner_rows"].sum()) != 500:
    raise ValueError(
        "Comment-evidence states do not partition all positive winner rows."
    )

positive_winner_comment_summary

,comment_evidence_state,runner_rows,provisional_races,unequal_distance_rows,minimum_raw_ovr_btn,maximum_raw_ovr_btn
0,explicit_amended_result,480,479,4,0.05,30.25
1,blank_comment,17,17,0,0.05,8.00
2,no_explicit_amendment_evidence,3,3,0,0.20,1.50


### Residual positive winner rows

**Grain of the preceding table:** one summary row per exploratory source-comment evidence state.

Explicit source comments explain 480 of the 500 positive-distance winner rows as amended results.

The remaining rows comprise:

* 17 rows with blank comments;
* 3 rows with nonblank comments but no amendment keyword detected.

Blank or unmatched comments do not prove that the result was not amended. The keyword classification may miss alternative wording, abbreviated comments or amendment evidence recorded elsewhere in the race.

The next review table inspects all 20 residual rows with source lineage and race context before deciding whether positive winner distances can be governed as an amended-result state.


In [17]:
# Inspect every positive-distance winner row without explicit amendment wording.
#
# Input grain:
#   One positive-distance winner runner row from
#   `positive_winner_comment_profile`.
#
# Output grain:
#   One residual runner row classified as either:
#       - `blank_comment`; or
#       - `no_explicit_amendment_evidence`.
#
# Purpose:
#   Explicit amendment wording explains 480 of the 500 positive winner rows.
#   This table exposes the remaining 20 rows so they are not silently treated
#   as equivalent to the explained population.
#
# Important:
#   - Raw source comments and distance values remain unchanged.
#   - The absence of a keyword match is not treated as evidence that the
#     official result was unchanged.
#   - No row is labelled as a source error.
#   - Physical source row order remains lineage only.

positive_winner_residual_rows = (
    positive_winner_comment_profile.loc[
        positive_winner_comment_profile["comment_evidence_state"]
        != "explicit_amended_result"
    ]
    .sort_values(
        ["date", "course", "off", "source_rowid"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Validate the residual count against the preceding summary:
# 17 blank comments plus 3 comments without an explicit amendment keyword.
if len(positive_winner_residual_rows) != 20:
    raise ValueError(
        "Expected 20 residual positive winner rows, "
        f"but found {len(positive_winner_residual_rows)}."
    )

display(
    positive_winner_residual_rows[
        [
            "source_rowid",
            "date",
            "course",
            "off",
            "race_id",
            "race_name",
            "type",
            "horse",
            "raw_pos",
            "raw_ovr_btn",
            "raw_btn",
            "comment_evidence_state",
            "raw_comment",
        ]
    ]
)

,source_rowid,date,course,off,race_id,race_name,type,horse,raw_pos,raw_ovr_btn,raw_btn,comment_evidence_state,raw_comment
0,354325,2017-04-11,Saint-Cloud (FR),12:47,673268,Prix du Pont de Flandre (Handicap) (4yo+) (Turf),Flat,Landjunge (GER),1,0.20,0.20,blank_comment,
1,621406,2018-10-21,Keeneland (USA),9:57,714713,Rood & Riddle Dowager Stakes (3yo+ Fillies & ...,Flat,Vexatious (USA),1,0.30,0.30,blank_comment,
2,791730,2019-10-14,Gulfstream Park West (USA),9:09,742692,Maiden Special Weight (Maiden) (2yo Fillies) (...,Flat,Cheermeister (USA),1,2.00,2.00,blank_comment,
3,854844,2020-04-09,Gulfstream Park (USA),6:30,755633,Claiming Race (Claimer) (3yo+) (Main Track) (D...,Flat,Congrats This (USA),1,0.30,0.30,blank_comment,
4,855119,2020-04-11,Gulfstream Park (USA),8:52,755716,Maiden Special Weight (Maiden) (3yo+) (Main Tr...,Flat,Colonel Liam (USA),1,2.00,2.00,blank_comment,
5,858575,2020-05-13,Will Rogers Downs (USA),11:45,757124,Maiden Claiming Race (3yo+ Fillies & Mares) (D...,Flat,Flying Lindy (USA),1,0.30,0.30,blank_comment,
6,859130,2020-05-28,Fonner Park (USA),12:42,758096,Claiming Race (3yo+ Fillies & Mares) (Main Tra...,Flat,Our Anabelle (USA),1,0.50,0.50,blank_comment,
7,872614,2020-08-10,Ballinrobe (IRE),6:45,764111,Connollys RED MILLS Irish EBF Auction Maiden H...,Hurdle,Rippon Lodge (IRE),1,1.25,1.25,no_explicit_amendment_evidence,Led - clear 2 out - reduced lead and edged lef...
8,895703,2020-09-22,Auteuil (FR),4:55,767702,Prix Marittimo (Hurdle) (Conditions) (4yo+) (T...,Hurdle,Brekdance Bilberry (FR),1,3.00,3.00,blank_comment,
9,1075280,2021-09-18,Laurel Park (USA),9:18,794079,Frank J De Francis Memorial Dash Stakes (3yo+...,Flat,Jalen Journey (USA),1,0.75,0.75,blank_comment,


### Race context for unexplained positive winner values

Explicit amended-result comments explain 480 of the 500 positive-distance winner rows.

Twenty rows remain without explicit amendment evidence:

* 17 have blank source comments;
* 3 have ordinary descriptive comments that appear consistent with the horse winning.

These rows cannot safely be assigned to the amended-result category.

The next inspection retrieves every runner in the 20 affected provisional races. The key questions are:

1. Does another runner in the race carry zero in one or both distance fields?
2. Do the distance values form a coherent sequence beginning from another runner?
3. Are there duplicate positive finishing positions?
4. Does physical source row order reveal anything material?
5. Are particular jurisdictional conventions involved?

No semantic status is assigned until the complete race rows have been reviewed.


In [19]:
# Retrieve complete race context for the 20 positive-winner rows that lack
# explicit amended-result evidence.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One source runner row belonging to one of the 20 selected provisional races.
#
# Purpose:
#   Positive values on official winner rows may have several explanations.
#   Reviewing the complete race allows us to test whether another runner carries
#   the zero state, whether margins follow a different finishing sequence, or
#   whether a jurisdiction-specific convention may be present.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - `source_rowid` preserves source lineage and physical row order only.
#   - Raw `pos`, `ovr_btn`, `btn` and `comment` remain unchanged.
#   - No row is classified as erroneous or amended in this cell.

# Build the distinct provisional-race keys represented by the 20 residual rows.
positive_winner_residual_race_keys = (
    positive_winner_residual_rows[
        ["date", "course", "off"]
    ]
    .drop_duplicates()
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Confirm that each residual winner row currently belongs to a distinct
# provisional race.
if len(positive_winner_residual_race_keys) != 20:
    raise ValueError(
        "Expected 20 distinct residual provisional races, "
        f"but found {len(positive_winner_residual_race_keys)}."
    )

# Construct a parameterised condition for the 20 candidate race identities.
residual_race_condition = " OR ".join(
    ["(date = ? AND course = ? AND off = ?)"]
    * len(positive_winner_residual_race_keys)
)

residual_race_parameters = [
    value
    for race_key in positive_winner_residual_race_keys.itertuples(
        index=False,
        name=None,
    )
    for value in race_key
]

positive_winner_residual_context_query = f"""
SELECT
    rowid AS source_rowid,
    date,
    course,
    off,
    race_id,
    race_name,
    type,
    horse,
    pos AS raw_pos,
    typeof(pos) AS pos_storage_class,
    ovr_btn AS raw_ovr_btn,
    typeof(ovr_btn) AS ovr_btn_storage_class,
    btn AS raw_btn,
    typeof(btn) AS btn_storage_class,
    ran AS raw_ran,
    num AS raw_num,
    comment AS raw_comment
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND ({residual_race_condition})
ORDER BY
    date,
    course,
    off,
    source_rowid
"""

# Execute through SQLite's read-only URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    positive_winner_residual_context = pd.read_sql_query(
        positive_winner_residual_context_query,
        connection,
        params=residual_race_parameters,
    )

# Confirm that context was returned for all 20 selected provisional races.
returned_residual_race_count = (
    positive_winner_residual_context[
        ["date", "course", "off"]
    ]
    .drop_duplicates()
    .shape[0]
)

if returned_residual_race_count != 20:
    raise ValueError(
        "Expected context for 20 residual provisional races, "
        f"but found {returned_residual_race_count}."
    )

# Produce a compact race-level diagnostic before printing every runner.
#
# Output grain:
#   One summary row per selected provisional race.
positive_winner_residual_race_summary = (
    positive_winner_residual_context
    .assign(
        numeric_ovr_btn=lambda frame: frame[
            "ovr_btn_storage_class"
        ].isin(["integer", "real"]),
        numeric_btn=lambda frame: frame[
            "btn_storage_class"
        ].isin(["integer", "real"]),
    )
    .groupby(
        ["date", "course", "off"],
        as_index=False,
        dropna=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        pos_one_rows=("raw_pos", lambda values: int((values == 1).sum())),
        ovr_btn_zero_rows=(
            "raw_ovr_btn",
            lambda values: int((values == 0).sum()),
        ),
        btn_zero_rows=(
            "raw_btn",
            lambda values: int((values == 0).sum()),
        ),
        minimum_raw_ovr_btn=("raw_ovr_btn", "min"),
        maximum_raw_ovr_btn=("raw_ovr_btn", "max"),
        minimum_raw_btn=("raw_btn", "min"),
        maximum_raw_btn=("raw_btn", "max"),
    )
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("Residual positive-winner race summary:")
display(positive_winner_residual_race_summary)

# Display each race separately for detailed review.
for race_key, race_rows in positive_winner_residual_context.groupby(
    ["date", "course", "off"],
    sort=False,
    dropna=False,
):
    race_date, race_course, race_off = race_key

    print(f"\n{race_date} — {race_course} — {race_off}")

    display(
        race_rows[
            [
                "source_rowid",
                "horse",
                "raw_pos",
                "raw_ovr_btn",
                "raw_btn",
                "raw_ran",
                "raw_num",
                "raw_comment",
            ]
        ].reset_index(drop=True)
    )

TypeError: agg function failed [how->min,dtype->object]

In [20]:
# Retrieve complete race context for the 20 positive-winner rows that lack
# explicit amended-result evidence.
#
# Input grain:
#   One governed source runner row.
#
# Output grains:
#   `positive_winner_residual_context`:
#       one source runner row from one of the 20 selected provisional races.
#
#   `positive_winner_residual_race_summary`:
#       one summary row per selected provisional race.
#
# Purpose:
#   Positive values on official winner rows may have several explanations.
#   Reviewing complete race context allows us to test whether another runner
#   carries the zero state, whether margins follow a different sequence, or
#   whether a jurisdiction-specific convention may be present.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - `source_rowid` preserves source lineage and physical row order only.
#   - Raw `pos`, `ovr_btn`, `btn` and `comment` remain unchanged.
#   - Numeric-only copies are created separately for summary calculations.
#   - No row is classified as erroneous or amended in this cell.

# Build the distinct provisional-race keys represented by the 20 residual rows.
positive_winner_residual_race_keys = (
    positive_winner_residual_rows[
        ["date", "course", "off"]
    ]
    .drop_duplicates()
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Confirm that each residual winner row currently belongs to a distinct
# provisional race.
if len(positive_winner_residual_race_keys) != 20:
    raise ValueError(
        "Expected 20 distinct residual provisional races, "
        f"but found {len(positive_winner_residual_race_keys)}."
    )

# Construct a parameterised SQL condition for the 20 candidate race identities.
residual_race_condition = " OR ".join(
    ["(date = ? AND course = ? AND off = ?)"]
    * len(positive_winner_residual_race_keys)
)

residual_race_parameters = [
    value
    for race_key in positive_winner_residual_race_keys.itertuples(
        index=False,
        name=None,
    )
    for value in race_key
]

positive_winner_residual_context_query = f"""
SELECT
    rowid AS source_rowid,
    date,
    course,
    off,
    race_id,
    race_name,
    type,
    horse,
    pos AS raw_pos,
    typeof(pos) AS pos_storage_class,
    ovr_btn AS raw_ovr_btn,
    typeof(ovr_btn) AS ovr_btn_storage_class,
    btn AS raw_btn,
    typeof(btn) AS btn_storage_class,
    ran AS raw_ran,
    num AS raw_num,
    comment AS raw_comment
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND ({residual_race_condition})
ORDER BY
    date,
    course,
    off,
    source_rowid
"""

# Execute through SQLite's read-only URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    positive_winner_residual_context = pd.read_sql_query(
        positive_winner_residual_context_query,
        connection,
        params=residual_race_parameters,
    )

# Confirm that context was returned for all 20 selected provisional races.
returned_residual_race_count = (
    positive_winner_residual_context[
        ["date", "course", "off"]
    ]
    .drop_duplicates()
    .shape[0]
)

if returned_residual_race_count != 20:
    raise ValueError(
        "Expected context for 20 residual provisional races, "
        f"but found {returned_residual_race_count}."
    )

# Create numeric-only analytical copies for range summaries.
#
# Raw source values remain unchanged in `raw_ovr_btn` and `raw_btn`.
# Text sentinels such as `-` become missing values only in these derived
# columns so pandas can calculate numeric minima and maxima safely.
positive_winner_residual_context["numeric_ovr_btn"] = pd.to_numeric(
    positive_winner_residual_context["raw_ovr_btn"],
    errors="coerce",
)

positive_winner_residual_context["numeric_btn"] = pd.to_numeric(
    positive_winner_residual_context["raw_btn"],
    errors="coerce",
)

# Produce a compact race-level diagnostic.
#
# Output grain:
#   One summary row per selected provisional race.
positive_winner_residual_race_summary = (
    positive_winner_residual_context
    .groupby(
        ["date", "course", "off"],
        as_index=False,
        dropna=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        pos_one_rows=(
            "raw_pos",
            lambda values: int((values == 1).sum()),
        ),
        ovr_btn_zero_rows=(
            "numeric_ovr_btn",
            lambda values: int((values == 0).sum()),
        ),
        btn_zero_rows=(
            "numeric_btn",
            lambda values: int((values == 0).sum()),
        ),
        minimum_numeric_ovr_btn=("numeric_ovr_btn", "min"),
        maximum_numeric_ovr_btn=("numeric_ovr_btn", "max"),
        minimum_numeric_btn=("numeric_btn", "min"),
        maximum_numeric_btn=("numeric_btn", "max"),
        ovr_btn_text_rows=(
            "ovr_btn_storage_class",
            lambda values: int((values == "text").sum()),
        ),
        btn_text_rows=(
            "btn_storage_class",
            lambda values: int((values == "text").sum()),
        ),
    )
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

print("Residual positive-winner race summary:")
display(positive_winner_residual_race_summary)

# Display each race separately for detailed review.
for race_key, race_rows in positive_winner_residual_context.groupby(
    ["date", "course", "off"],
    sort=False,
    dropna=False,
):
    race_date, race_course, race_off = race_key

    print(f"\n{race_date} — {race_course} — {race_off}")

    display(
        race_rows[
            [
                "source_rowid",
                "horse",
                "raw_pos",
                "raw_ovr_btn",
                "raw_btn",
                "raw_ran",
                "raw_num",
                "raw_comment",
            ]
        ].reset_index(drop=True)
    )

Residual positive-winner race summary:


,date,course,off,runner_rows,pos_one_rows,ovr_btn_zero_rows,btn_zero_rows,minimum_numeric_ovr_btn,maximum_numeric_ovr_btn,minimum_numeric_btn,maximum_numeric_btn,ovr_btn_text_rows,btn_text_rows
0,2017-04-11,Saint-Cloud (FR),12:47,16,1,1,1,0.0,28.25,0.00,20.00,0,0
1,2018-10-21,Keeneland (USA),9:57,9,1,1,1,0.0,24.50,0.00,8.75,0,0
2,2019-10-14,Gulfstream Park West (USA),9:09,12,1,1,1,0.0,29.25,0.00,12.75,0,0
3,2020-04-09,Gulfstream Park (USA),6:30,7,1,1,1,0.0,19.75,0.00,5.50,0,0
4,2020-04-11,Gulfstream Park (USA),8:52,10,1,1,1,0.0,16.00,0.00,4.00,1,1
5,2020-05-13,Will Rogers Downs (USA),11:45,9,1,1,1,0.0,46.50,0.00,26.00,0,0
6,2020-05-28,Fonner Park (USA),12:42,8,1,1,1,0.0,10.50,0.00,3.25,0,0
7,2020-08-10,Ballinrobe (IRE),6:45,10,1,1,1,0.0,142.25,0.00,81.00,3,3
8,2020-09-22,Auteuil (FR),4:55,18,1,1,1,0.0,40.50,0.00,12.00,6,6
9,2021-09-18,Laurel Park (USA),9:18,6,1,1,1,0.0,9.25,0.00,4.75,0,0



2017-04-11 — Saint-Cloud (FR) — 12:47


,source_rowid,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,354310,Zalzali (FR),16,28.25,20,16,15,
1,354312,Burnside (FR),15,8.25,0.75,16,16,
2,354313,Bunook (GB),14,7.5,0.2,16,7,
3,354314,Teodash (IRE),13,7.5,2,16,14,
4,354315,Vincento (GB),12,5.5,0.3,16,11,
5,354316,Dagobert Duke (GB),11,5.25,0.2,16,9,
6,354317,Alzarodesvillerets (FR),10,5,0.3,16,2,
7,354318,Gris Noir (FR),9,4.75,0.75,16,6,
8,354320,Prince Nomad (FR),8,4,1,16,1,
9,354321,Le Pin (FR),6,2.75,0.75,16,10,



2018-10-21 — Keeneland (USA) — 9:57


...

,source_rowid,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1515474,Kool And Girly (FR),PU,-,-,11,10,
1,1515490,Jalda Des Obeaux (FR),PU,-,-,11,1,
2,1515491,Apres Bridget (IRE),8,15.25,2.5,11,6,
3,1515492,Walkyria (FR),7,12.75,3,11,8,
4,1515515,Fiction Du Berlais (FR),DSQ,0,0,11,2,
5,1515516,Bakala (FR),1,2.5,2.5,11,12,
6,1515517,Kristal Du Seuil (FR),2,5,2.5,11,7,
7,1515518,Sanntama DOroux (FR),3,6.5,1.5,11,9,
8,1515519,Calliandra (FR),4,9.5,3,11,4,
9,1515520,Guetta (FR),5,9.5,0.1,11,5,



2025-08-31 — Del Mar (USA) — 1:39


,source_rowid,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1726673,Reef Runner (USA),2,0,0,12,12,
1,1726674,Motorious (GB),1,0.05,0.05,12,10,
2,1726675,Beyond Brilliant (USA),3,1,1,12,1,
3,1726676,No Nay Hudson (IRE),4,1.25,0.3,12,2,
4,1726677,Book Smart (USA),5,1.5,0.2,12,6,
5,1726678,Queen Maxima (USA),6,2,0.5,12,11,
6,1726679,Sorrento Sky (IRE),7,2,0.05,12,4,
7,1726680,Sumter (USA),8,2.75,0.75,12,8,
8,1726681,Boss Sully (USA),9,4,1.25,12,7,
9,1726682,Virat (USA),10,4.25,0.2,12,5,



2026-05-01 — Palermo — 19:05


,source_rowid,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1836914,Colorado Del Monte (ARG),3,0,0,7,7,
1,1836915,Emotion Rate (ARG),1,0.1,0.1,7,1,
2,1836916,Paulo Sil (ARG),2,0.5,0.3,7,3,
3,1836917,Cardo Castilla (ARG),4,1.5,1,7,6,
4,1836918,Desert Voice (ARG),5,1.5,0.2,7,5,
5,1836919,Thomas Shelby (ARG),6,13.5,12,7,2,
6,1836920,Blue Star (ARG),7,19.5,6,7,4,


### Positive winner values and original finishing order

Complete race context explains 19 of the 20 positive-winner residual races.

In each of those 19 races:

* the official winner has a positive `ovr_btn` and `btn`;
* another runner carries zero in both fields; and
* the remaining distance sequence is consistent with the zero-valued runner occupying the original physical winning position.

The zero-valued runner may later appear as:

* second;
* third; or
* `DSQ`.

This establishes that explicit amendment wording in `comment` is not required for the distance fields to retain an original physical result.

Across the positive-winner population, the defensible interpretation is therefore:

> Positive distance values on a runner with official `pos = 1` ordinarily indicate that the horse was promoted to first after the physical finish, while `ovr_btn` and `btn` retained the original finishing-distance sequence.

The source fields can consequently describe different result stages:

* `pos` can represent the amended official placing;
* `ovr_btn` can remain anchored to the original physical winner;
* `btn` can remain anchored to the original physical finishing sequence.

This means that sorting solely by official `pos` will produce false arithmetic contradictions in amended-result races.

One residual race remains unresolved:

`2023-12-23 + Gulfstream Park (USA) + 9:36`

In that race:

* no runner carries zero in either distance field;
* the official winner has `ovr_btn = 4.25` and `btn = 4.25`;
* the official second has `ovr_btn = 4` and `btn = 4`;
* raw positions omit position 5;
* a runner has raw position 9 despite `ran = 8`.

The observed rows are compatible with an omitted or removed original winner, but the source alone does not prove that explanation. The race must remain an unresolved positive-winner exception pending external or manual verification.


### Manual verification: Gulfstream Park, 23 December 2023

The unresolved race was checked against external result evidence.

Race identity:

`2023-12-23 + Gulfstream Park (USA) + 9:36`

Race:

`Mr. Prospector Stakes`

The external result establishes that:

* nine runners competed;
* Sibelius finished first;
* Gilmore finished second;
* Dreaming Of Kona finished third;
* Long Range Toddy finished fourth;
* Great Navigator finished fifth; and
* the source extract omitted Great Navigator.

The source contains eight runner rows even though retained finishing positions extend to ninth place. The missing fifth-place runner explains the discontinuity in the supplied result population.

This is not an amended-winner case. Sibelius was the physical and official winner.

The positive distance values on the retained winner row cannot be interpreted safely from the incomplete source rows alone. The supplied distance sequence is misaligned because at least one result row is absent.

### Governance decision

The immutable raw source must remain unchanged.

The manually verified missing runner should be captured separately as governed supplementary result data and included during future database processing.

The supplementary record must preserve:

* candidate race identity;
* supplied `race_id`;
* missing horse identity;
* verified finishing position;
* external evidence and locator;
* access date;
* verification status;
* confidence;
* supplementation method; and
* a clear distinction between source-present and externally supplemented data.

The database build should therefore:

1. load all immutable source runner rows;
2. add the governed supplementary runner record;
3. retain a provenance flag showing that the row was absent from the source;
4. process the completed race population through the result and beaten-distance pipeline; and
5. prevent the supplementary row from being mistaken for an original Raceform source row.

No missing distance values should be invented. Any verified distances may be added only where the external evidence explicitly supports them.

Manual-verification status:

`captured`

Permitted future database action:

`add governed supplementary runner record during processing`

The raw race remains a documented source omission, while the processed database may contain the externally verified complete result with explicit provenance.


## 3. Zero distance values on later numeric finishers

**Input grain:** one governed source runner row.

**Review-table grain:** one summary row per observed zero-distance relationship.

The bounded raw profile found:

* 371 runners with raw numeric `pos > 1` and `ovr_btn = 0`;
* 3,121 runners with raw numeric `pos > 1` and `btn = 0`.

These values may reflect:

* dead heats or tied finishing positions;
* amended official placings;
* original physical winners later moved down the result;
* source rounding conventions;
* incomplete race records; or
* unresolved source anomalies.

Zero is not interpreted automatically as a winner marker or dead-heat marker.

The first step profiles the two zero states jointly against raw finishing position and distance equality.


In [21]:
# Profile zero-distance states among later positive numeric finishers.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One summary row per raw finishing position and joint zero-state pattern.
#
# Purpose:
#   The initial profile found later finishers with zero in one or both distance
#   fields. This query separates those states before any race-level explanation
#   is attempted.
#
# Important:
#   - Raw `pos`, `ovr_btn` and `btn` remain unchanged.
#   - Only physically numeric SQLite values are included.
#   - Zero is not assigned a semantic meaning in this cell.
#   - Runner-row findings remain distinct from later race-level findings.

later_finisher_zero_query = f"""
WITH later_numeric_finishers AS (
    SELECT
        pos AS raw_pos,
        ovr_btn AS raw_ovr_btn,
        btn AS raw_btn,
        CASE
            WHEN ovr_btn = 0 AND btn = 0
                THEN 'both_zero'
            WHEN ovr_btn = 0 AND btn > 0
                THEN 'ovr_btn_zero_only'
            WHEN ovr_btn > 0 AND btn = 0
                THEN 'btn_zero_only'
            ELSE 'neither_zero'
        END AS zero_state
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
      AND typeof(pos) IN ('integer', 'real')
      AND pos > 1
      AND typeof(ovr_btn) IN ('integer', 'real')
      AND typeof(btn) IN ('integer', 'real')
)
SELECT
    raw_pos,
    zero_state,
    COUNT(*) AS runner_rows,
    COUNT(DISTINCT raw_ovr_btn) AS distinct_raw_ovr_btn_values,
    COUNT(DISTINCT raw_btn) AS distinct_raw_btn_values,
    MIN(raw_ovr_btn) AS minimum_raw_ovr_btn,
    MAX(raw_ovr_btn) AS maximum_raw_ovr_btn,
    MIN(raw_btn) AS minimum_raw_btn,
    MAX(raw_btn) AS maximum_raw_btn,
    SUM(raw_ovr_btn = raw_btn) AS exactly_equal_rows
FROM later_numeric_finishers
WHERE zero_state <> 'neither_zero'
GROUP BY
    raw_pos,
    zero_state
ORDER BY
    raw_pos,
    zero_state
"""

# Execute through SQLite's read-only URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    later_finisher_zero_profile = pd.read_sql_query(
        later_finisher_zero_query,
        connection,
    )

# Validate the profile against the earlier runner-row totals.
observed_ovr_btn_zero_rows = int(
    later_finisher_zero_profile.loc[
        later_finisher_zero_profile["zero_state"].isin(
            ["both_zero", "ovr_btn_zero_only"]
        ),
        "runner_rows",
    ].sum()
)

observed_btn_zero_rows = int(
    later_finisher_zero_profile.loc[
        later_finisher_zero_profile["zero_state"].isin(
            ["both_zero", "btn_zero_only"]
        ),
        "runner_rows",
    ].sum()
)

if observed_ovr_btn_zero_rows != 371:
    raise ValueError(
        "Later-finisher ovr_btn-zero count differs from the bounded profile: "
        f"{observed_ovr_btn_zero_rows:,}"
    )

if observed_btn_zero_rows != 3_121:
    raise ValueError(
        "Later-finisher btn-zero count differs from the bounded profile: "
        f"{observed_btn_zero_rows:,}"
    )

# Display the complete position-by-zero-state profile.
later_finisher_zero_profile

,raw_pos,zero_state,runner_rows,distinct_raw_ovr_btn_values,distinct_raw_btn_values,minimum_raw_ovr_btn,maximum_raw_ovr_btn,minimum_raw_btn,maximum_raw_btn,exactly_equal_rows
0,2,both_zero,293,1,1,0.00,0.00,0,0,293
1,2,btn_zero_only,363,35,1,0.05,20.00,0,0,0
2,3,both_zero,34,1,1,0.00,0.00,0,0,34
3,3,btn_zero_only,432,48,1,0.20,32.50,0,0,0
4,4,both_zero,11,1,1,0.00,0.00,0,0,11
5,4,btn_zero_only,459,59,1,0.25,46.00,0,0,0
6,5,both_zero,6,1,1,0.00,0.00,0,0,6
7,5,btn_zero_only,412,59,1,0.50,42.75,0,0,0
8,6,both_zero,8,1,1,0.00,0.00,0,0,8
9,6,btn_zero_only,299,71,1,0.75,57.00,0,0,0


### Initial zero-state result

The later-finisher zero population separates completely into two states:

| Zero state          | Runner rows |
| ------------------- | ----------: |
| `both_zero`         |         371 |
| `btn_zero_only`     |       2,750 |
| `ovr_btn_zero_only` |           0 |

Every later numeric finisher with `ovr_btn = 0` also has `btn = 0`.

This indicates that zero in `ovr_btn` is not occurring as an isolated arithmetic value. It identifies a runner that also carries the zero state in the adjacent-margin field.

The two observed states must therefore be investigated separately:

* `both_zero` may identify the original physical winner in an amended result, a tied zero reference, or another result-order exception;
* `btn_zero_only` may represent a zero margin between adjacent runners, including dead heats, rounded margins or source defects.

No semantic classification is assigned yet.


In [22]:
# Profile complete race structure for later finishers carrying zero in both
# beaten-distance fields.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One summary row per provisional race containing at least one numeric
#   finisher with raw pos > 1 and both distance fields equal to zero.
#
# Purpose:
#   `both_zero` may represent the original physical winner in an amended
#   result, a tied zero reference, or another source exception. This cell
#   measures the surrounding race structure before inspecting examples.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - Raw source fields remain unchanged.
#   - Official `pos` and physical distance order are not assumed to agree.
#   - This is a race-level structural profile, not a final classification.

both_zero_race_profile_query = f"""
WITH governed_rows AS (
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        pos AS raw_pos,
        ovr_btn AS raw_ovr_btn,
        btn AS raw_btn,
        comment AS raw_comment
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
),
both_zero_race_keys AS (
    SELECT DISTINCT
        date,
        course,
        off
    FROM governed_rows
    WHERE typeof(raw_pos) IN ('integer', 'real')
      AND raw_pos > 1
      AND typeof(raw_ovr_btn) IN ('integer', 'real')
      AND typeof(raw_btn) IN ('integer', 'real')
      AND raw_ovr_btn = 0
      AND raw_btn = 0
)
SELECT
    rows.date,
    rows.course,
    rows.off,
    COUNT(*) AS runner_rows,
    SUM(
        typeof(rows.raw_pos) IN ('integer', 'real')
        AND rows.raw_pos = 1
    ) AS official_winner_rows,
    SUM(
        typeof(rows.raw_pos) IN ('integer', 'real')
        AND rows.raw_pos > 1
        AND typeof(rows.raw_ovr_btn) IN ('integer', 'real')
        AND typeof(rows.raw_btn) IN ('integer', 'real')
        AND rows.raw_ovr_btn = 0
        AND rows.raw_btn = 0
    ) AS later_both_zero_rows,
    MIN(
        CASE
            WHEN typeof(rows.raw_pos) IN ('integer', 'real')
             AND rows.raw_pos > 1
             AND typeof(rows.raw_ovr_btn) IN ('integer', 'real')
             AND typeof(rows.raw_btn) IN ('integer', 'real')
             AND rows.raw_ovr_btn = 0
             AND rows.raw_btn = 0
            THEN rows.raw_pos
        END
    ) AS minimum_later_zero_pos,
    MAX(
        CASE
            WHEN typeof(rows.raw_pos) IN ('integer', 'real')
             AND rows.raw_pos > 1
             AND typeof(rows.raw_ovr_btn) IN ('integer', 'real')
             AND typeof(rows.raw_btn) IN ('integer', 'real')
             AND rows.raw_ovr_btn = 0
             AND rows.raw_btn = 0
            THEN rows.raw_pos
        END
    ) AS maximum_later_zero_pos,
    SUM(
        typeof(rows.raw_pos) = 'text'
        AND rows.raw_pos = 'DSQ'
    ) AS dsq_rows,
    SUM(
        COALESCE(rows.raw_comment, '') LIKE '%originally%'
        OR COALESCE(rows.raw_comment, '') LIKE '%awarded%'
        OR COALESCE(rows.raw_comment, '') LIKE '%disqualif%'
        OR COALESCE(rows.raw_comment, '') LIKE '%demoted%'
        OR COALESCE(rows.raw_comment, '') LIKE '%promoted%'
        OR COALESCE(rows.raw_comment, '') LIKE '%placed%'
    ) AS explicit_amendment_comment_rows
FROM governed_rows AS rows
INNER JOIN both_zero_race_keys AS race_keys
    ON rows.date = race_keys.date
   AND rows.course = race_keys.course
   AND rows.off = race_keys.off
GROUP BY
    rows.date,
    rows.course,
    rows.off
ORDER BY
    rows.date,
    rows.course,
    rows.off
"""

# Execute through SQLite's read-only URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    both_zero_race_profile = pd.read_sql_query(
        both_zero_race_profile_query,
        connection,
    )

# Validate that the race-level profile accounts for all 371 both-zero rows.
observed_both_zero_rows = int(
    both_zero_race_profile["later_both_zero_rows"].sum()
)

if observed_both_zero_rows != 371:
    raise ValueError(
        "Race-level both-zero count differs from the runner-level profile: "
        f"{observed_both_zero_rows:,}"
    )

# Summarise the principal race structures.
both_zero_structure_summary = pd.DataFrame(
    [
        {
            "measure": "provisional races",
            "value": len(both_zero_race_profile),
        },
        {
            "measure": "races with one later both-zero runner",
            "value": int(
                both_zero_race_profile["later_both_zero_rows"].eq(1).sum()
            ),
        },
        {
            "measure": "races with multiple later both-zero runners",
            "value": int(
                both_zero_race_profile["later_both_zero_rows"].gt(1).sum()
            ),
        },
        {
            "measure": "races containing DSQ",
            "value": int(
                both_zero_race_profile["dsq_rows"].gt(0).sum()
            ),
        },
        {
            "measure": "races with explicit amendment wording",
            "value": int(
                both_zero_race_profile[
                    "explicit_amendment_comment_rows"
                ].gt(0).sum()
            ),
        },
        {
            "measure": "highest observed later zero position",
            "value": int(
                both_zero_race_profile[
                    "maximum_later_zero_pos"
                ].max()
            ),
        },
    ]
)

display(both_zero_structure_summary)

# Show the most structurally unusual races first.
display(
    both_zero_race_profile.sort_values(
        [
            "later_both_zero_rows",
            "maximum_later_zero_pos",
            "dsq_rows",
        ],
        ascending=[False, False, False],
        kind="stable",
    )
    .head(30)
    .reset_index(drop=True)
)

,measure,value
0,provisional races,341
1,races with one later both-zero runner,337
2,races with multiple later both-zero runners,4
3,races containing DSQ,0
4,races with explicit amendment wording,322
5,highest observed later zero position,17


,date,course,off,runner_rows,official_winner_rows,later_both_zero_rows,minimum_later_zero_pos,maximum_later_zero_pos,dsq_rows,explicit_amendment_comment_rows
0,2026-05-25,San Isidro,19:22,14,1,13,2,14,0,0
1,2026-05-25,San Isidro,22:25,12,1,11,2,12,0,0
2,2016-12-17,Haydock,3:15,9,1,7,2,8,0,0
3,2022-02-06,St Moritz (SWI),12:30,4,1,3,2,4,0,0
4,2019-05-04,Churchill Downs (USA),11:50,19,1,1,17,17,0,17
5,2015-04-18,Keeneland (USA),9:11,12,1,1,10,10,0,10
6,2021-05-28,Stratford,5:10,9,1,1,7,7,0,7
7,2015-01-08,Gulfstream Park (USA),7:07,8,1,1,6,6,0,6
8,2015-09-24,Maisons-Laffitte (FR),3:00,12,1,1,6,6,0,6
9,2018-08-29,Deauville (FR),1:50,16,1,1,6,6,0,6


In [23]:
# Retrieve complete runner context for the four races containing multiple
# later finishers with zero in both distance fields, plus one clear
# amendment-wording comparison race.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One source runner row from one of five selected provisional races.
#
# Purpose:
#   The race-level profile revealed two potentially different structures:
#
#   1. races where almost every later finisher carries zero in both fields;
#   2. races where one later zero-holder appears alongside explicit result-
#      amendment wording.
#
#   Complete context is needed before these structures can be classified.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - Raw source values are displayed unchanged.
#   - Source row order is preserved for lineage only.
#   - Zero is not automatically interpreted as a dead heat or amended result.

selected_both_zero_races = [
    ("2026-05-25", "San Isidro", "19:22"),
    ("2026-05-25", "San Isidro", "22:25"),
    ("2016-12-17", "Haydock", "3:15"),
    ("2022-02-06", "St Moritz (SWI)", "12:30"),
    ("2019-05-04", "Churchill Downs (USA)", "11:50"),
]

# Construct a parameterised condition for the selected candidate race keys.
selected_race_condition = " OR ".join(
    ["(date = ? AND course = ? AND off = ?)"] * len(selected_both_zero_races)
)

selected_race_parameters = [
    value
    for race_key in selected_both_zero_races
    for value in race_key
]

selected_both_zero_context_query = f"""
SELECT
    rowid AS source_rowid,
    date,
    course,
    off,
    race_id,
    race_name,
    type,
    horse,
    pos AS raw_pos,
    typeof(pos) AS pos_storage_class,
    ovr_btn AS raw_ovr_btn,
    typeof(ovr_btn) AS ovr_btn_storage_class,
    btn AS raw_btn,
    typeof(btn) AS btn_storage_class,
    ran AS raw_ran,
    num AS raw_num,
    comment AS raw_comment
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND ({selected_race_condition})
ORDER BY
    date,
    course,
    off,
    source_rowid
"""

# Execute against the immutable SQLite source through its read-only URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    selected_both_zero_context = pd.read_sql_query(
        selected_both_zero_context_query,
        connection,
        params=selected_race_parameters,
    )

# Confirm that all five selected provisional races were returned.
returned_selected_race_count = (
    selected_both_zero_context[
        ["date", "course", "off"]
    ]
    .drop_duplicates()
    .shape[0]
)

if returned_selected_race_count != len(selected_both_zero_races):
    raise ValueError(
        "Expected context for "
        f"{len(selected_both_zero_races)} selected races, but found "
        f"{returned_selected_race_count}."
    )

# Display each race separately so its distance sequence and comments can be
# reviewed without mixing runner rows from different races.
for race_key, race_rows in selected_both_zero_context.groupby(
    ["date", "course", "off"],
    sort=True,
    dropna=False,
):
    race_date, race_course, race_off = race_key

    print(f"\n{race_date} — {race_course} — {race_off}")

    display(
        race_rows[
            [
                "source_rowid",
                "race_name",
                "horse",
                "raw_pos",
                "raw_ovr_btn",
                "raw_btn",
                "raw_ran",
                "raw_num",
                "raw_comment",
            ]
        ].reset_index(drop=True)
    )


2016-12-17 — Haydock — 3:15


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,310645,32Red.com Handicap Hurdle,Le Rocher (FR),2,0,0,9,2,Prominent early - midfield after 4th - trackin...
1,310646,32Red.com Handicap Hurdle,El Terremoto (FR),3,0,0,9,3,Held up - ridden briefly after 4 out - finishe...
2,310647,32Red.com Handicap Hurdle,Draytonian (IRE),4,0,0,9,4,In rear early - tracked leaders after 4 out - ...
3,310648,32Red.com Handicap Hurdle,Sharp Response (IRE),5,0,0,9,9,Held up early - tracking leaders after 4th - s...
4,310649,32Red.com Handicap Hurdle,Super Sam (GB),6,0,0,9,7,Prominent early - in lead after 4th - weakened...
5,310650,32Red.com Handicap Hurdle,Cooking Fat (GB),7,0,0,9,5,In touch early - still handy after 4th - proba...
6,310651,32Red.com Handicap Hurdle,Great Tempo (FR),8,0,0,9,8,Prominent early - narrow leader after 4th - st...
7,310652,32Red.com Handicap Hurdle,Lettheriverrundry (IRE),PU,-,-,9,6,In touch early - in midfield after 4th - pulle...
8,310653,32Red.com Handicap Hurdle,Clyne (GB),1,0,0,9,1,In lead early - still prominent after 4th - to...



2019-05-04 — Churchill Downs (USA) — 11:50


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,706975,Kentucky Derby presented by Woodford Reserve ...,Long Range Toddy (USA),16,18.5,1.5,19,18,Chased leader - ridden 3f out - hung right and...
1,706977,Kentucky Derby presented by Woodford Reserve ...,Cutting Humor (USA),10,9.5,0.75,19,10,Towards rear of midfield - headway on wide out...
2,706978,Kentucky Derby presented by Woodford Reserve ...,By My Standards (USA),11,11.5,2,19,3,Dwelt - towards rear - ridden and some headway...
3,706979,Kentucky Derby presented by Woodford Reserve ...,Vekoma (USA),12,15,3.5,19,6,In touch until lost place quickly 3f out - soo...
4,706980,Kentucky Derby presented by Woodford Reserve ...,Bodexpress (USA),13,15.25,0.3,19,21,Tracked leaders - ridden 3f out - slightly out...
5,706981,Kentucky Derby presented by Woodford Reserve ...,Tax (USA),14,15.5,0.2,19,2,Towards rear of midfield - ridden and effort o...
6,706982,Kentucky Derby presented by Woodford Reserve ...,Roadster (USA),15,17,1.5,19,17,Always towards rear - finished 16th - placed 15th
7,706983,Kentucky Derby presented by Woodford Reserve ...,Spinoff (USA),18,18.5,0.05,19,19,Midfield towards outer - ridden over 3f out - ...
8,706984,Kentucky Derby presented by Woodford Reserve ...,Gray Magician (USA),19,26.75,8.25,19,4,Towards rear - ridden and headway into midfiel...
9,707002,Kentucky Derby presented by Woodford Reserve ...,Win Win Win (USA),9,8.75,3.25,19,14,Towards rear - not clear run approaching 2f ou...



2022-02-06 — St Moritz (SWI) — 12:30


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1139307,Grosser Preis Longines (Conditions) (4yo+) (Snow),De La Fayette (GER),3,0,0,4,4,
1,1139308,Grosser Preis Longines (Conditions) (4yo+) (Snow),Arktisz (FR),2,0,0,4,7,
2,1139309,Grosser Preis Longines (Conditions) (4yo+) (Snow),Mordred (IRE),1,0,0,4,9,Made all - shaken up entering final furlong - ...
3,1139318,Grosser Preis Longines (Conditions) (4yo+) (Snow),Toscano (FR),4,0,0,4,6,



2026-05-25 — San Isidro — 19:22


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1850245,Gran Premio de Potrancas Stakes (Filllies) (Turf),Gran Gotera (ARG),1,0,0,14,3,
1,1850246,Gran Premio de Potrancas Stakes (Filllies) (Turf),Gota Clara Sos (ARG),2,0,0,14,1,
2,1850247,Gran Premio de Potrancas Stakes (Filllies) (Turf),Gunakali (ARG),3,0,0,14,5,
3,1850248,Gran Premio de Potrancas Stakes (Filllies) (Turf),Live Your Life (ARG),4,0,0,14,8,
4,1850249,Gran Premio de Potrancas Stakes (Filllies) (Turf),Unica Forma (ARG),5,0,0,14,4,
5,1850250,Gran Premio de Potrancas Stakes (Filllies) (Turf),Full Question (ARG),6,0,0,14,9,
6,1850251,Gran Premio de Potrancas Stakes (Filllies) (Turf),Just Dance (ARG),7,0,0,14,2,
7,1850252,Gran Premio de Potrancas Stakes (Filllies) (Turf),Salema (ARG),8,0,0,14,13,
8,1850253,Gran Premio de Potrancas Stakes (Filllies) (Turf),Inter Bombaza (ARG),9,0,0,14,6,
9,1850254,Gran Premio de Potrancas Stakes (Filllies) (Turf),Deep Feeling (ARG),10,0,0,14,14,



2026-05-25 — San Isidro — 22:25


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1850289,Gran Premio 25 de Mayo Copa Dr. Enrique Oliver...,Magnum Fifth (ARG),1,0,0,12,1,
1,1850290,Gran Premio 25 de Mayo Copa Dr. Enrique Oliver...,Curious Song (ARG),2,0,0,12,9,
2,1850291,Gran Premio 25 de Mayo Copa Dr. Enrique Oliver...,Honest Boy (ARG),3,0,0,12,4,
3,1850292,Gran Premio 25 de Mayo Copa Dr. Enrique Oliver...,Out Of The Blue (BRZ),4,0,0,12,10,
4,1850293,Gran Premio 25 de Mayo Copa Dr. Enrique Oliver...,Equal Mostaza (ARG),5,0,0,12,6,
5,1850294,Gran Premio 25 de Mayo Copa Dr. Enrique Oliver...,Full Keynote (ARG),6,0,0,12,3,
6,1850295,Gran Premio 25 de Mayo Copa Dr. Enrique Oliver...,Neat Colt (ARG),7,0,0,12,7,
7,1850296,Gran Premio 25 de Mayo Copa Dr. Enrique Oliver...,Time To Think (ARG),8,0,0,12,11,
8,1850297,Gran Premio 25 de Mayo Copa Dr. Enrique Oliver...,Castanon Rim (ARG),9,0,0,12,5,
9,1850298,Gran Premio 25 de Mayo Copa Dr. Enrique Oliver...,Holy Holy Rim (ARG),10,0,0,12,12,


### Complete-context review of multiple-zero races

The selected race contexts establish two distinct uses of `ovr_btn = 0` and `btn = 0`.

#### Race-wide zero distance coverage

Four reviewed races contain zero in both fields for every numeric finisher:

* Haydock, 17 December 2016;
* St Moritz, 6 February 2022;
* San Isidro, 25 May 2026 at 19:22;
* San Isidro, 25 May 2026 at 22:25.

Their finishing positions remain differentiated, but the distance fields contain no usable margin information.

These races must not be interpreted as mass dead heats. They represent unavailable or failed beaten-distance capture encoded as numeric zero.

#### Isolated zero-holder after an amended result

The 2019 Kentucky Derby contains one later-placed runner with zero in both fields:

* Maximum Security physically finished first;
* the official result placed him seventeenth;
* Country House became the official winner;
* the distance sequence remained anchored to Maximum Security as the physical winner.

This is consistent with the positive-winner examples reviewed earlier.

### Provisional rule

A later numeric finisher carrying `0 / 0` has no single source-wide meaning.

Interpretation requires race context:

* where all numeric finishers carry `0 / 0`, beaten-distance coverage is unavailable;
* where one runner carries `0 / 0` within an otherwise populated sequence, that runner may be the original physical winner in an amended result;
* other structures remain to be classified separately.

No zero value should be interpreted independently of the surrounding race.


In [24]:
# Classify races containing later numeric finishers with zero in both distance
# fields according to the coverage of their complete numeric-finisher population.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One summary row per provisional race containing at least one later numeric
#   finisher with `ovr_btn = 0` and `btn = 0`.
#
# Purpose:
#   Complete-context review showed that `both_zero` can represent either:
#
#   - race-wide unavailable distance capture; or
#   - an isolated zero-holder inside an otherwise populated distance sequence.
#
#   This cell quantifies those structures across the complete source.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - Raw source values remain unchanged.
#   - Numeric finishers include all rows with a positive numeric `pos`.
#   - This cell classifies observed coverage structure only.
#   - It does not yet prove that every isolated zero-holder reflects an amended
#     result.

both_zero_coverage_query = f"""
WITH governed_rows AS (
    SELECT
        date,
        course,
        off,
        pos AS raw_pos,
        ovr_btn AS raw_ovr_btn,
        btn AS raw_btn
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
),
race_profile AS (
    SELECT
        date,
        course,
        off,
        SUM(
            typeof(raw_pos) IN ('integer', 'real')
            AND raw_pos > 0
        ) AS numeric_finisher_rows,
        SUM(
            typeof(raw_pos) IN ('integer', 'real')
            AND raw_pos > 0
            AND typeof(raw_ovr_btn) IN ('integer', 'real')
            AND typeof(raw_btn) IN ('integer', 'real')
            AND raw_ovr_btn = 0
            AND raw_btn = 0
        ) AS numeric_both_zero_rows,
        SUM(
            typeof(raw_pos) IN ('integer', 'real')
            AND raw_pos > 1
            AND typeof(raw_ovr_btn) IN ('integer', 'real')
            AND typeof(raw_btn) IN ('integer', 'real')
            AND raw_ovr_btn = 0
            AND raw_btn = 0
        ) AS later_both_zero_rows
    FROM governed_rows
    GROUP BY
        date,
        course,
        off
)
SELECT
    date,
    course,
    off,
    numeric_finisher_rows,
    numeric_both_zero_rows,
    later_both_zero_rows,
    CASE
        WHEN numeric_both_zero_rows = numeric_finisher_rows
            THEN 'all_numeric_finishers_both_zero'
        WHEN numeric_both_zero_rows = 1
            THEN 'single_zero_holder'
        ELSE 'multiple_partial_zero_holders'
    END AS both_zero_coverage_state
FROM race_profile
WHERE later_both_zero_rows > 0
ORDER BY
    date,
    course,
    off
"""

# Execute against the immutable source through the read-only SQLite URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    both_zero_coverage_profile = pd.read_sql_query(
        both_zero_coverage_query,
        connection,
    )

# Confirm that all 371 later both-zero runner rows remain represented.
observed_later_both_zero_rows = int(
    both_zero_coverage_profile["later_both_zero_rows"].sum()
)

if observed_later_both_zero_rows != 371:
    raise ValueError(
        "Coverage classification does not reproduce the 371 later "
        f"both-zero rows: {observed_later_both_zero_rows:,}"
    )

# Summarise the number of races and runner rows represented by each observed
# coverage structure.
both_zero_coverage_summary = (
    both_zero_coverage_profile
    .groupby(
        "both_zero_coverage_state",
        as_index=False,
        dropna=False,
    )
    .agg(
        provisional_races=("date", "size"),
        numeric_finisher_rows=("numeric_finisher_rows", "sum"),
        numeric_both_zero_rows=("numeric_both_zero_rows", "sum"),
        later_both_zero_rows=("later_both_zero_rows", "sum"),
        minimum_numeric_finishers=("numeric_finisher_rows", "min"),
        maximum_numeric_finishers=("numeric_finisher_rows", "max"),
    )
    .sort_values(
        "provisional_races",
        ascending=False,
        kind="stable",
    )
    .reset_index(drop=True)
)

display(both_zero_coverage_summary)

# Display any partial multi-zero structures separately because they do not fit
# either the race-wide-unavailable or isolated-zero-holder pattern.
display(
    both_zero_coverage_profile.loc[
        both_zero_coverage_profile[
            "both_zero_coverage_state"
        ].eq("multiple_partial_zero_holders")
    ].reset_index(drop=True)
)

,both_zero_coverage_state,provisional_races,numeric_finisher_rows,numeric_both_zero_rows,later_both_zero_rows,minimum_numeric_finishers,maximum_numeric_finishers
0,single_zero_holder,322,3083,322,322,3,21
1,multiple_partial_zero_holders,15,132,30,15,4,15
2,all_numeric_finishers_both_zero,4,38,38,34,4,14


,date,course,off,numeric_finisher_rows,numeric_both_zero_rows,later_both_zero_rows,both_zero_coverage_state
0,2015-01-15,Wolverhampton (AW),6:40,7,2,1,multiple_partial_zero_holders
1,2016-10-28,Uttoxeter,2:35,6,2,1,multiple_partial_zero_holders
2,2017-10-17,Leicester,5:30,12,2,1,multiple_partial_zero_holders
3,2022-04-07,Aintree,2:20,6,2,1,multiple_partial_zero_holders
4,2022-07-13,Brighton,1:00,4,2,1,multiple_partial_zero_holders
5,2024-05-21,Huntingdon,8:10,9,2,1,multiple_partial_zero_holders
6,2024-07-28,Uttoxeter,2:25,12,2,1,multiple_partial_zero_holders
7,2024-09-09,Wolverhampton (AW),4:55,5,2,1,multiple_partial_zero_holders
8,2024-11-16,Wolverhampton (AW),8:00,12,2,1,multiple_partial_zero_holders
9,2025-04-06,Gavea (BRZ),7:35,4,2,1,multiple_partial_zero_holders


### Coverage-state profile

The 371 later `both_zero` rows occur across 341 provisional races:

| Coverage state                    | Provisional races | Later `both_zero` rows |
| --------------------------------- | ----------------: | ---------------------: |
| `single_zero_holder`              |               322 |                    322 |
| `multiple_partial_zero_holders`   |                15 |                     15 |
| `all_numeric_finishers_both_zero` |                 4 |                     34 |

The four race-wide cases have already been identified as unavailable distance capture.

The 322 single-zero-holder races form the principal amended-result candidate population.

The remaining 15 races each contain:

* one official winner with `0 / 0`; and
* one later numeric finisher with `0 / 0`.

These races do not fit either the race-wide-unavailable pattern or the isolated original-winner pattern. Because the population is bounded at 15 races, complete race-level review is appropriate before assigning a semantic rule.


In [27]:
# Retrieve complete runner context for all 15 races classified as having
# multiple partial zero-holders.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One source runner row from one of the 15 selected provisional races.
#
# Purpose:
#   Each selected race contains exactly two numeric finishers with zero in
#   both distance fields: normally the official winner and one later finisher.
#   Complete context is required to distinguish dead heats, amended results,
#   source defects and any other repeated convention.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - Raw source values and comments remain unchanged.
#   - Source row order is preserved for lineage only.
#   - No semantic classification is assigned in this cell.
#   - The complete bounded population of 15 races is displayed, not sampled.
# Show complete comment text in notebook tables.


#
# Purpose:
#   Pandas truncates long text columns by default, which hides the evidence
#   needed for manual race-level review.
#
# Effect:
#   Display only. No source values or dataframe contents are changed.

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)


# Extract the 15 provisional race identities identified by the source-wide
# coverage classification.
partial_zero_race_keys = (
    both_zero_coverage_profile.loc[
        both_zero_coverage_profile[
            "both_zero_coverage_state"
        ].eq("multiple_partial_zero_holders"),
        ["date", "course", "off"],
    ]
    .drop_duplicates()
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Validate the bounded population before querying complete race context.
if len(partial_zero_race_keys) != 15:
    raise ValueError(
        "Expected 15 multiple-partial-zero races, "
        f"but found {len(partial_zero_race_keys)}."
    )

# Build a parameterised SQLite condition for the 15 candidate race keys.
partial_zero_race_condition = " OR ".join(
    ["(date = ? AND course = ? AND off = ?)"]
    * len(partial_zero_race_keys)
)

partial_zero_race_parameters = [
    value
    for race_key in partial_zero_race_keys.itertuples(
        index=False,
        name=None,
    )
    for value in race_key
]

partial_zero_context_query = f"""
SELECT
    rowid AS source_rowid,
    date,
    course,
    off,
    race_id,
    race_name,
    type,
    horse,
    pos AS raw_pos,
    typeof(pos) AS pos_storage_class,
    ovr_btn AS raw_ovr_btn,
    typeof(ovr_btn) AS ovr_btn_storage_class,
    btn AS raw_btn,
    typeof(btn) AS btn_storage_class,
    ran AS raw_ran,
    num AS raw_num,
    comment AS raw_comment
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND ({partial_zero_race_condition})
ORDER BY
    date,
    course,
    off,
    source_rowid
"""

# Execute against the immutable source through the read-only SQLite URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    partial_zero_context = pd.read_sql_query(
        partial_zero_context_query,
        connection,
        params=partial_zero_race_parameters,
    )

# Confirm that complete context was returned for all 15 provisional races.
returned_partial_zero_race_count = (
    partial_zero_context[
        ["date", "course", "off"]
    ]
    .drop_duplicates()
    .shape[0]
)

if returned_partial_zero_race_count != 15:
    raise ValueError(
        "Expected context for 15 multiple-partial-zero races, "
        f"but found {returned_partial_zero_race_count}."
    )

# Display each complete race independently so the zero-holders, official
# positions, comments and surrounding distance sequence can be reviewed.
for race_key, race_rows in partial_zero_context.groupby(
    ["date", "course", "off"],
    sort=True,
    dropna=False,
):
    race_date, race_course, race_off = race_key

    print(f"\n{race_date} — {race_course} — {race_off}")

    display(
        race_rows[
            [
                "source_rowid",
                "race_name",
                "horse",
                "raw_pos",
                "raw_ovr_btn",
                "raw_btn",
                "raw_ran",
                "raw_num",
                "raw_comment",
            ]
        ].reset_index(drop=True)
    )


2015-01-15 — Wolverhampton (AW) — 6:40


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,4524,Bet In Play At Coral Handicap (Tapeta),Halfsin (IRE),7,92.5,81,7,1,Prominent until ridden and weakened over 2f out - eased (trainer said gelding had a breathing problem)(op 12/1)
1,4525,Bet In Play At Coral Handicap (Tapeta),Hanalei Bay (IRE),6,11.5,3.25,7,4,Led 1f - chased leader until over 5f out - remained handy - ridden over 2f out - weakened over 1f out(op 11/2 tchd 6/1 and 9/2)
2,4526,Bet In Play At Coral Handicap (Tapeta),Chapter And Verse (IRE),5,8.25,1,7,5,Slowly into stride - recovered to lead after 1f - ridden and headed well over 1f out - weakened inside final furlong(op 6/1 tchd 8/1)
3,4527,Bet In Play At Coral Handicap (Tapeta),Off The Pulse (GB),4,7.25,2.25,7,6,Tracked leaders - raced keenly - went 2nd over 5f out until led well over 1f out - soon ridden and headed - weakened inside final furlong(tchd 2/1)
4,4529,Bet In Play At Coral Handicap (Tapeta),Monsea (IRE),3,5,5,7,3,Held up - ridden over 2f out - ran on inside final furlong - not trouble leaders(op 33/1)
5,4530,Bet In Play At Coral Handicap (Tapeta),Berlusca (IRE),1,0,0,7,2,Held up - headway over 1f out - ridden to chase winner and edged left 1f out - carried right towards finish - ran on to dead-heat post - dead-heated for 1st - awarded race outright(op 5/1)
6,4538,Bet In Play At Coral Handicap (Tapeta),Warfare (GB),2,0,0,7,7,Chased leaders - led over 1f out - ridden and edged left inside final furlong - edged right towards finish - joined post - dead-heated for 1st - disqualified and placed 2nd(op 7/2)



2016-10-28 — Uttoxeter — 2:35


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,288324,Sun Bets Novices Hurdle (Northern Novice Stayers Series Qualifier.),Derrick DAnjou (IRE),6,37.75,21,6,4,Chased leader - effort 7th - lost place before next - behind when mistake 2 out - soon tailed off
1,288325,Sun Bets Novices Hurdle (Northern Novice Stayers Series Qualifier.),Desert Retreat (IRE),5,16.75,7,6,5,In rear - chased leaders 5th - driven 7th - lost place before next(op 11/1)
2,288326,Sun Bets Novices Hurdle (Northern Novice Stayers Series Qualifier.),Cash Again (FR),4,9.75,9,6,1,Took keen hold - tracked leaders - 2nd 7th - weakened last (jockey said that the gelding ran too freely in the early stages.)(op 8/11 tchd 4/6)
3,288327,Sun Bets Novices Hurdle (Northern Novice Stayers Series Qualifier.),Chelsea Flyer (IRE),3,0.75,0.75,6,2,Took keen hold - not fluent 3rd - headway 7th - upsides at last - keeping on inside when squeezed out near line(op 3/1 tchd 7/2)
4,288328,Sun Bets Novices Hurdle (Northern Novice Stayers Series Qualifier.),Cracking Find (IRE),1,0,0,6,3,Best away - clear until after 2nd - stayed on from 2 out - carried left final 75yds - joined post - awarded race(op 40/1)
5,288329,Sun Bets Novices Hurdle (Northern Novice Stayers Series Qualifier.),One Forty Seven (IRE),2,0,0,6,7,Held up - tracked leaders 5th - upsides 2 out - hung left between last 2 - edged left last 75yds - dead-heated for 1st - disqualified and placed 2nd(op 13/2 tchd 4/1)



2017-10-17 — Leicester — 5:30


...

,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1760692,Try Unibets Smartview Racecards Handicap,Maywedance (IRE),1,0,0,13,12,In touch with leaders - dropped to midfield after 2f - not clear run over 2f out - soon switched right and good headway - led narrowly inside final furlong - keeping on when slightly hampered towards finish - joined post - finished dead-heat 1st - awarded race(op 7/1)
1,1760693,Try Unibets Smartview Racecards Handicap,Electric Bass (GB),2,0,0,13,13,Took keen hold - prominent - led narrowly over 2f out - soon bumped - headed but pressed winner inside final furlong - bumped rival towards finish - kept on and forced dead-heat - finished dead-heat 1st - placed 2nd - interference
2,1760694,Try Unibets Smartview Racecards Handicap,Ricardo Phillips (GB),3,1.25,1.25,13,9,In touch with leaders - headway over 2f out - ran on and challenging over 1f out - no extra inside final furlong(op 11/1)
3,1760695,Try Unibets Smartview Racecards Handicap,Damascus Steel (IRE),4,2,0.75,13,2,Towards rear - headway on outer over 3f out - kept on and went fourth final 110yds(op 17/2 tchd 7/1 and tchd 9/1)
4,1760696,Try Unibets Smartview Racecards Handicap,Chambers (IRE),5,4,2,13,6,Prominent on outer - bumped rival 2f out - outpaced from over 1f out(tchd 18/5)
5,1760697,Try Unibets Smartview Racecards Handicap,Buck Barrow (GB),6,4.5,0.5,13,14,Towards rear - some headway from over 1f out - nearest finish(tchd 25/1)
6,1760698,Try Unibets Smartview Racecards Handicap,Uzincso (GB),7,5.25,0.75,13,3,In touch with leaders - bit short of room over 2f out - not clear run under 2f out - soon outpaced(op 7/1 tchd 9/1)
7,1760699,Try Unibets Smartview Racecards Handicap,Soldiers Chorus (IRE),8,7.75,2.5,13,11,Towards rear throughout(op 16/1)
8,1760700,Try Unibets Smartview Racecards Handicap,Koko Blue (GB),9,9.25,1.5,13,8,Led early - prominent - weakened over 1f out(tchd 40/1)
9,1760701,Try Unibets Smartview Racecards Handicap,Grey Phoenix (IRE),10,11.75,2.5,13,7,Pulled hard - always behind(op 10/1 tchd 12/1)



2025-12-17 — Deauville — 17:20


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1780222,Prix du Sentier du Littoral (Maiden) (3yo) (All-Weather Track) (Polytrack),Chevaldor (GB),2,0,0,15,9,
1,1780223,Prix du Sentier du Littoral (Maiden) (3yo) (All-Weather Track) (Polytrack),Highfield World (FR),1,0,0,15,7,
2,1780224,Prix du Sentier du Littoral (Maiden) (3yo) (All-Weather Track) (Polytrack),Grand Chelem (FR),3,0.2,0.2,15,10,
3,1780225,Prix du Sentier du Littoral (Maiden) (3yo) (All-Weather Track) (Polytrack),Rinascero (IRE),4,1.5,1.25,15,3,
4,1780226,Prix du Sentier du Littoral (Maiden) (3yo) (All-Weather Track) (Polytrack),Takeo (GER),5,3.5,2,15,11,
5,1780227,Prix du Sentier du Littoral (Maiden) (3yo) (All-Weather Track) (Polytrack),Zodd (IRE),6,4.5,1,15,8,
6,1780228,Prix du Sentier du Littoral (Maiden) (3yo) (All-Weather Track) (Polytrack),Opi Daqui (FR),7,10.5,6,15,5,
7,1780229,Prix du Sentier du Littoral (Maiden) (3yo) (All-Weather Track) (Polytrack),Dhalem (FR),8,12.5,2,15,15,
8,1780230,Prix du Sentier du Littoral (Maiden) (3yo) (All-Weather Track) (Polytrack),Catherines Gift (IRE),9,13.25,0.75,15,12,
9,1780231,Prix du Sentier du Littoral (Maiden) (3yo) (All-Weather Track) (Polytrack),Seven For All (FR),10,14,0.75,15,4,



2026-01-19 — Kempton (AW) — 13:08


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1793216,racingtv.com Handicap,Token Gesture (IRE),1,0,0,8,3,Prominent - ridden to lead inside final furlong - edged left towards finish - bumped final strides - joined post - awarded race outright(op 13/2 tchd 11/2)
1,1793217,racingtv.com Handicap,Apache Green (GB),2,0,0,8,1,In rear - ridden and headway up the centre over 1f out - hung right when bumped rival towards finish - forced dead-heat - placed 2nd - interference(op 7/1 tchd 13/2 and tchd 8/1)
2,1793218,racingtv.com Handicap,Mbappe (GB),3,0.75,0.75,8,6,Dwelt start - in rear - headway over 2f out - raced in third and keeping on when squeezed out and bumped rival towards finish(op 10/3)
3,1793219,racingtv.com Handicap,Villalobos (IRE),4,2,1.25,8,7,Led - headed inside final furlong - no extra final 110yds(op 10/1 tchd 9/1)
4,1793220,racingtv.com Handicap,Equion (GB),5,3.25,1.25,8,4,Midfield - headway over 1f out - no extra final 110yds(op 15/2 tchd 13/2 and tchd 8/1)
5,1793221,racingtv.com Handicap,Hopjes (SAF),6,4.5,1.25,8,8,Never better than midfield(op 28/1)
6,1793222,racingtv.com Handicap,Super Hit (FR),7,5.75,1.25,8,2,Travelled strongly - prominent - weakened over 1f out(op 6/4)
7,1793223,racingtv.com Handicap,Purple Sky (IRE),8,15.25,9.5,8,5,Always behind(op 50/1)


### Multiple partial zero-holders

Complete review resolved all 15 races containing exactly two numeric finishers with `ovr_btn = 0` and `btn = 0`.

#### Amended dead-heat results

Fourteen races represent a physical dead heat for first followed by an amended official result.

In these races:

* the two physical first-place finishers both retain `ovr_btn = 0` and `btn = 0`;
* one runner is recorded as official `pos = 1`;
* the other is recorded as official `pos = 2`; and
* the official separation arose through disqualification, demotion, interference or an award of the race.

Twelve races state this directly in the source comments.

The blank-comment Ascot and Deauville examples were confirmed through governed manual verification:

* `NB15-BTN-0002`;
* `NB15-BTN-0003`.

These races show that the distance fields may preserve a physical dead heat while `pos` records the amended official result.

#### Source-distance contradiction

One race does not follow that convention:

`2025-04-06 + Gavea (BRZ) + 7:35`

The source stores both Naturalizada and Gevrey-Chambertain at `0 / 0`. External results establish that Naturalizada won and Gevrey-Chambertain finished second, with a published winning margin of 16½ lengths.

This is therefore a defective source-distance sequence rather than a dead heat.

The contradiction is governed by:

`NB15-BTN-0004`

The immutable raw values must remain unchanged. The verified winning margin may be applied only through a governed downstream reconciliation layer.

### Resulting interpretation

The `multiple_partial_zero_holders` structure is highly diagnostic but not infallible:

* 14 of 15 races are amended physical dead heats;
* 1 of 15 is a verified source defect.

A pair of zero-distance finishers must therefore be interpreted using comments or governed external evidence rather than classified solely from the numeric pattern.


### Single zero-holder races

The largest `both_zero` group contains 322 races with exactly one numeric zero-holder.

In these races:

* one runner carries `ovr_btn = 0` and `btn = 0`;
* the remaining numeric finishers have populated nonzero distance values; and
* the zero-holder may occupy an official position later than first.

The reviewed Kentucky Derby example showed that this structure can preserve the original physical winner after an amended official result.

The next step measures how often the source comments explicitly explain the official placing change and identifies the residual cases that may require manual verification.


In [28]:
# Profile source-comment evidence for all 322 single-zero-holder races.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One summary row per single-zero-holder provisional race.
#
# Purpose:
#   The single-zero-holder structure is a strong amended-result candidate, but
#   it must not be treated as proof by itself. This cell checks whether source
#   comments explicitly describe disqualification, demotion, promotion,
#   awarded placings or an original finishing position.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - Raw comments and result fields remain unchanged.
#   - Comment matching is evidence discovery, not final semantic classification.
#   - Blank or unmatched comments remain residual cases for later review.

single_zero_race_keys = (
    both_zero_coverage_profile.loc[
        both_zero_coverage_profile[
            "both_zero_coverage_state"
        ].eq("single_zero_holder"),
        ["date", "course", "off"],
    ]
    .drop_duplicates()
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Validate the bounded race population before querying source comments.
if len(single_zero_race_keys) != 322:
    raise ValueError(
        "Expected 322 single-zero-holder races, "
        f"but found {len(single_zero_race_keys)}."
    )

# Build a parameterised SQLite condition for all selected race identities.
single_zero_race_condition = " OR ".join(
    ["(date = ? AND course = ? AND off = ?)"]
    * len(single_zero_race_keys)
)

single_zero_race_parameters = [
    value
    for race_key in single_zero_race_keys.itertuples(
        index=False,
        name=None,
    )
    for value in race_key
]

single_zero_comment_query = f"""
SELECT
    rowid AS source_rowid,
    date,
    course,
    off,
    horse,
    pos AS raw_pos,
    ovr_btn AS raw_ovr_btn,
    btn AS raw_btn,
    comment AS raw_comment
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND ({single_zero_race_condition})
ORDER BY
    date,
    course,
    off,
    source_rowid
"""

# Execute against the immutable source through the read-only SQLite URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    single_zero_comment_context = pd.read_sql_query(
        single_zero_comment_query,
        connection,
        params=single_zero_race_parameters,
    )

# Create a normalised comment copy for evidence-pattern matching only.
#
# The raw source comment remains preserved in `raw_comment`.
single_zero_comment_context["normalised_comment"] = (
    single_zero_comment_context["raw_comment"]
    .fillna("")
    .astype(str)
    .str.lower()
)

# Mark rows containing explicit result-amendment language.
single_zero_comment_context["explicit_amendment_evidence"] = (
    single_zero_comment_context["normalised_comment"].str.contains(
        r"\b("
        r"disqualif|"
        r"demot|"
        r"promot|"
        r"awarded|"
        r"placed\s+\d|"
        r"originally|"
        r"finished\s+\d|"
        r"relegat"
        r")",
        regex=True,
        na=False,
    )
)

# Mark blank comments separately from populated but unmatched comments.
single_zero_comment_context["blank_comment"] = (
    single_zero_comment_context["normalised_comment"].str.strip().eq("")
)

# Summarise evidence at provisional-race grain.
single_zero_comment_profile = (
    single_zero_comment_context
    .groupby(
        ["date", "course", "off"],
        as_index=False,
        dropna=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        explicit_amendment_comment_rows=(
            "explicit_amendment_evidence",
            "sum",
        ),
        blank_comment_rows=("blank_comment", "sum"),
        populated_comment_rows=(
            "blank_comment",
            lambda values: int((~values).sum()),
        ),
    )
)

# Assign an evidence state without interpreting unmatched races automatically.
single_zero_comment_profile["comment_evidence_state"] = (
    "explicit_amendment_evidence"
)

single_zero_comment_profile.loc[
    single_zero_comment_profile[
        "explicit_amendment_comment_rows"
    ].eq(0)
    & single_zero_comment_profile["populated_comment_rows"].eq(0),
    "comment_evidence_state",
] = "all_comments_blank"

single_zero_comment_profile.loc[
    single_zero_comment_profile[
        "explicit_amendment_comment_rows"
    ].eq(0)
    & single_zero_comment_profile["populated_comment_rows"].gt(0),
    "comment_evidence_state",
] = "populated_without_explicit_amendment"
    
# Validate that every single-zero-holder race is represented exactly once.
if len(single_zero_comment_profile) != 322:
    raise ValueError(
        "Expected 322 race-level comment-profile rows, "
        f"but found {len(single_zero_comment_profile)}."
    )

# Summarise the complete comment-evidence partition.
single_zero_comment_summary = (
    single_zero_comment_profile
    .groupby(
        "comment_evidence_state",
        as_index=False,
        dropna=False,
    )
    .agg(
        provisional_races=("date", "size"),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values(
        "provisional_races",
        ascending=False,
        kind="stable",
    )
    .reset_index(drop=True)
)

display(single_zero_comment_summary)

# Display the residual races that lack explicit amendment wording.
display(
    single_zero_comment_profile.loc[
        ~single_zero_comment_profile[
            "comment_evidence_state"
        ].eq("explicit_amendment_evidence")
    ]
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

/tmp/ipykernel_375435/3088508376.py:99: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  single_zero_comment_context["normalised_comment"].str.contains(


,comment_evidence_state,provisional_races,runner_rows
0,explicit_amendment_evidence,304,3007
1,all_comments_blank,11,110
2,populated_without_explicit_amendment,7,64


,date,course,off,runner_rows,explicit_amendment_comment_rows,blank_comment_rows,populated_comment_rows,comment_evidence_state
0,2015-06-20,Ayr,2:20,6,0,0,6,populated_without_explicit_amendment
1,2016-03-12,Flemington (AUS),5:30,10,0,6,4,populated_without_explicit_amendment
2,2017-04-11,Saint-Cloud (FR),12:47,16,0,16,0,all_comments_blank
3,2018-10-21,Keeneland (USA),9:57,9,0,9,0,all_comments_blank
4,2019-10-14,Gulfstream Park West (USA),9:09,12,0,12,0,all_comments_blank
5,2020-04-09,Gulfstream Park (USA),6:30,7,0,7,0,all_comments_blank
6,2020-04-11,Gulfstream Park (USA),8:52,10,0,9,1,populated_without_explicit_amendment
7,2020-05-13,Will Rogers Downs (USA),11:45,9,0,9,0,all_comments_blank
8,2020-05-28,Fonner Park (USA),12:42,8,0,8,0,all_comments_blank
9,2021-09-18,Laurel Park (USA),9:18,6,0,5,1,populated_without_explicit_amendment


In [29]:
# Retrieve complete context for the seven single-zero-holder races whose
# comments are populated but contain no amendment wording matched by the
# current regular expression.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One source runner row from one of the seven selected provisional races.
#
# Purpose:
#   These races may contain explanatory wording that the bounded amendment
#   pattern failed to recognise. Complete comments must be reviewed before
#   deciding whether external manual verification is necessary.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - Raw comments and distance values remain unchanged.
#   - The seven-race population is complete, not sampled.
#   - No race is classified automatically in this cell.

# Select the complete bounded population of populated-comment residual races.
populated_single_zero_residual_keys = (
    single_zero_comment_profile.loc[
        single_zero_comment_profile[
            "comment_evidence_state"
        ].eq("populated_without_explicit_amendment"),
        ["date", "course", "off"],
    ]
    .drop_duplicates()
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Validate the expected bounded population before retrieving full context.
if len(populated_single_zero_residual_keys) != 7:
    raise ValueError(
        "Expected seven populated-comment residual races, "
        f"but found {len(populated_single_zero_residual_keys)}."
    )

# Build a parameterised SQLite condition for the seven candidate race keys.
populated_single_zero_condition = " OR ".join(
    ["(date = ? AND course = ? AND off = ?)"]
    * len(populated_single_zero_residual_keys)
)

populated_single_zero_parameters = [
    value
    for race_key in populated_single_zero_residual_keys.itertuples(
        index=False,
        name=None,
    )
    for value in race_key
]

populated_single_zero_context_query = f"""
SELECT
    rowid AS source_rowid,
    date,
    course,
    off,
    race_id,
    race_name,
    type,
    horse,
    pos AS raw_pos,
    typeof(pos) AS pos_storage_class,
    ovr_btn AS raw_ovr_btn,
    typeof(ovr_btn) AS ovr_btn_storage_class,
    btn AS raw_btn,
    typeof(btn) AS btn_storage_class,
    ran AS raw_ran,
    num AS raw_num,
    comment AS raw_comment
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND ({populated_single_zero_condition})
ORDER BY
    date,
    course,
    off,
    source_rowid
"""

# Execute against the immutable source through its read-only SQLite URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    populated_single_zero_context = pd.read_sql_query(
        populated_single_zero_context_query,
        connection,
        params=populated_single_zero_parameters,
    )

# Confirm that complete context was returned for all seven races.
returned_populated_residual_races = (
    populated_single_zero_context[
        ["date", "course", "off"]
    ]
    .drop_duplicates()
    .shape[0]
)

if returned_populated_residual_races != 7:
    raise ValueError(
        "Expected complete context for seven populated-comment residual "
        f"races, but found {returned_populated_residual_races}."
    )

# Display each race independently with complete comment text.
for race_key, race_rows in populated_single_zero_context.groupby(
    ["date", "course", "off"],
    sort=True,
    dropna=False,
):
    race_date, race_course, race_off = race_key

    print(f"\n{race_date} — {race_course} — {race_off}")

    display(
        race_rows[
            [
                "source_rowid",
                "race_name",
                "horse",
                "raw_pos",
                "raw_ovr_btn",
                "raw_btn",
                "raw_ran",
                "raw_num",
                "raw_comment",
            ]
        ].reset_index(drop=True)
    )


2015-06-20 — Ayr — 2:20


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,72840,Scottish Sun On Sunday Handicap,Mister Sea Wolf (IRE),6,11.75,10,6,1,Chased leaders - ridden over 2f out - weakened well over 1f out
1,72842,Scottish Sun On Sunday Handicap,Mukhayyam (GB),4,0.75,0.1,6,2,Held up - smooth headway over 2f out - ridden over 1f out - kept on same pace inside final furlong
2,72881,Scottish Sun On Sunday Handicap,Spring Offensive (IRE),5,1.75,1,6,4,Took keen hold - pressed leader - ridden over 2f out - outpaced when checked approaching final furlong - soon no danger
3,72897,Scottish Sun On Sunday Handicap,Little Lady Katie (IRE),2,0,0,6,6,Made all - ridden over 2f out - drifted right inside final furlong - held on gamely - finished first - placed second(op 11/4)
4,72898,Scottish Sun On Sunday Handicap,Intiwin (IRE),1,0.1,0.1,6,3,Tracked leaders - effort and ridden 2f out - every chance when carried right and baulked inside final furlong - just held - finished second - placed first(op 5/1)
5,72899,Scottish Sun On Sunday Handicap,Get Knotted (IRE),3,0.5,0.5,6,5,Took keen hold - held up in touch - ridden and outpaced over 2f out - rallied approaching final furlong - kept on strongly towards finish(op 7/2)



2016-03-12 — Flemington (AUS) — 5:30


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,178559,Australian Cup (3yo+) (Turf),Ivanhowe (GER),5,1.5,0.1,10,5,Raced as Our Ivanhowe
1,178560,Australian Cup (3yo+) (Turf),Extra Zero (AUS),6,1.75,0.1,10,3,
2,178561,Australian Cup (3yo+) (Turf),Happy Trails (AUS),7,1.75,0.2,10,1,
3,178562,Australian Cup (3yo+) (Turf),Bow Creek (IRE),8,2,0.1,10,6,Slow to start - in rear - ridden over 2f out - kept on steadily but never able to challenge
4,178563,Australian Cup (3yo+) (Turf),Suavito (NZ),9,3,1,10,10,
5,178564,Australian Cup (3yo+) (Turf),Almoonqith (USA),10,3,0.1,10,7,
6,178600,Australian Cup (3yo+) (Turf),Fenway (AUS),4,1.5,0.1,10,11,
7,178622,Australian Cup (3yo+) (Turf),Rising Romance (NZ),3,1.25,1.25,10,9,
8,178624,Australian Cup (3yo+) (Turf),Awesome Rock (AUS),2,0,0,10,8,Finished first - placed second
9,178646,Australian Cup (3yo+) (Turf),Preferment (NZ),1,0.1,0.1,10,2,Finished second - placed first



2020-04-11 — Gulfstream Park (USA) — 8:52


...

,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1600789,Hung Shui Kiu Handicap (3yo+) (Course C) (Turf),Helene Warrior (IRE),12,14.75,2.25,12,1,Raced keenly - prominent - wide around first turn - led after 1 1/2f - ridden 2f out - headed under 2f out - weakened over 1f out - eased inside final furlong
1,1600790,Hung Shui Kiu Handicap (3yo+) (Course C) (Turf),Tourbillon Prince (AUS),11,12.5,4.75,12,5,In touch - ridden 2 1/2f out - lost place 1 1/2f out - hampered over 1f out - soon beaten - eased inside final furlong
2,1600791,Hung Shui Kiu Handicap (3yo+) (Course C) (Turf),Second To None (NZ),10,7.75,2.5,12,11,Midfield on outside - ridden over 2f out - wide entering straight and lost place - weakened final furlong
3,1600792,Hung Shui Kiu Handicap (3yo+) (Course C) (Turf),Toy Soltero (CHI),9,5.25,0.3,12,10,Held up towards rear - ridden 1 1/2f out - brief effort 1f out - no extra final 150yds
4,1600793,Hung Shui Kiu Handicap (3yo+) (Course C) (Turf),Apache Pass (AUS),8,5,1,12,3,Started slowly - towards rear of midfield - ridden 2f out - driven and some headway over 1f out - no extra final furlong
5,1600794,Hung Shui Kiu Handicap (3yo+) (Course C) (Turf),Soleil Fighter (GB),7,4,1.75,12,9,Led - headed after 1 1/2f - chased leader - led under 2f out - ridden 1 1/2f out - headed over 1f out - driven and no extra final furlong
6,1600795,Hung Shui Kiu Handicap (3yo+) (Course C) (Turf),Silo (AUS),6,2.25,0.5,12,7,Held up towards rear - ridden 2f out - wide entering straight - driven and stayed on final furlong - nearest finish
7,1600796,Hung Shui Kiu Handicap (3yo+) (Course C) (Turf),Simply Maverick (AUS),5,1.75,0.3,12,2,Held up in rear - ridden 1 1/2f out - driven and kept on well final furlong - not clear run 50yds out - no further impression closing stages
8,1600797,Hung Shui Kiu Handicap (3yo+) (Course C) (Turf),I Can (NZ),4,1.5,0.1,12,8,Midfield - ridden and headway under 2f out - not clear run 1 1/2f out - driven and kept on final furlong - not going pace to challenge
9,1600798,Hung Shui Kiu Handicap (3yo+) (Course C) (Turf),Romantic Laos (NZ),3,1.25,1.25,12,6,Towards rear of midfield - ridden and headway under 2f out - driven and edged left over 1f out - kept on well final furlong - not going pace to challenge



2025-02-09 — Sha Tin (HK) — 9:50


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1632852,TVB The Fading Gold Handicap (3yo+) (Course C) (Turf),Casa Rochester (NZ),10,8.25,1.75,14,13,Always towards rear
1,1632853,TVB The Fading Gold Handicap (3yo+) (Course C) (Turf),Equatorial (USA),14,29.25,11,14,6,Towards rear of midfield on outside - ridden 2 1/2f out - weakened under 2f out
2,1632854,TVB The Fading Gold Handicap (3yo+) (Course C) (Turf),Mr Vigor (IRE),13,18.25,9.75,14,9,Towards rear of midfield - ridden under 2f out - weakened 1 1/2f out
3,1632855,TVB The Fading Gold Handicap (3yo+) (Course C) (Turf),Keefy (AUS),12,8.5,0.3,14,1,In touch - ridden under 2f out - weakened final furlong
4,1632856,TVB The Fading Gold Handicap (3yo+) (Course C) (Turf),Lost Child (NZ),11,8.25,0.05,14,11,Midfield on outside - ridden and effort 2f out - edged left over 1f out - weakened final furlong
5,1632857,TVB The Fading Gold Handicap (3yo+) (Course C) (Turf),Test For Love (IRE),9,6.5,1.25,14,10,Held up towards rear - switched wide entering straight - ridden 2f out - slightly hampered 1 1/2f out - kept on final furlong - never dangerous
6,1632858,TVB The Fading Gold Handicap (3yo+) (Course C) (Turf),Beauty Glory (GB),8,5.25,0.05,14,12,Led - soon clear - came back to field over 2f out - ridden under 2f out - headed 1 1/2f out - driven and no extra from over 1f out
7,1632860,TVB The Fading Gold Handicap (3yo+) (Course C) (Turf),Solid Impact (NZ),5,2.75,0.5,14,8,Held up towards rear - ridden 2f out - not clear run and angled wide 1 1/2f out - stayed on well final furlong - nearest finish
8,1632861,TVB The Fading Gold Handicap (3yo+) (Course C) (Turf),Storming Dragon (AUS),6,3.75,1,14,4,In touch - ridden and chased leaders 2f out - no extra inside final furlong
9,1632862,TVB The Fading Gold Handicap (3yo+) (Course C) (Turf),Masterofmyuniverse (GB),7,5,1.25,14,3,Held up in rear - switched wide entering straight - ridden 2f out - kept on final furlong - never dangerous



2025-08-01 — Saratoga (USA) — 9:36


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,raw_ran,raw_num,raw_comment
0,1713781,National Museum of Racing Hall of Fame Stakes (3yo) (Inner Turf),Zulu Kingdom (IRE),4,0,0,6,7,
1,1713782,National Museum of Racing Hall of Fame Stakes (3yo) (Inner Turf),Luther (GB),1,1.5,1.5,6,2,Clipped heels soon after start - in rear - squeezed up and bumped by rival entering first turn - headway on inner under 4f out - soon not clear run - ridden and headway on inner 1 1/2f out - went second inside final furlong - finished second - placed first
2,1713783,National Museum of Racing Hall of Fame Stakes (3yo) (Inner Turf),Clever Again (USA),2,3,1.5,6,5,Raced prominently - took closer order 3f out - ridden along over 1f out - outpaced by leaders inside final furlong - finished third - placed second
3,1713784,National Museum of Racing Hall of Fame Stakes (3yo) (Inner Turf),Tiz Dashing (USA),3,3.5,0.5,6,3,
4,1713785,National Museum of Racing Hall of Fame Stakes (3yo) (Inner Turf),Mi Bago (USA),5,6.5,3,6,1,
5,1713786,National Museum of Racing Hall of Fame Stakes (3yo) (Inner Turf),Tank (USA),6,23.75,17.25,6,8,


### Populated-comment single-zero residuals

All seven single-zero-holder races with populated comments but no initially matched amendment phrase were resolved.

Five races contain sufficient explanation within the source comments:

* Ayr, 20 June 2015;
* Flemington, 12 March 2016;
* Laurel Park, 18 September 2021;
* Happy Valley, 20 November 2024;
* Sha Tin, 9 February 2025.

In each race:

* the runner carrying `ovr_btn = 0` and `btn = 0` passed the post first;
* that runner was subsequently placed second;
* the official winner was promoted from second; and
* the promoted official winner retains a positive distance from the original physical winner.

The original pattern did not match these races because the comments used wording such as:

* `finished first - placed second`; and
* `finished second - placed first`.

Two further races required governed manual verification:

* `NB15-BTN-0005` — Gulfstream Park, 11 April 2020;
* `NB15-BTN-0006` — Saratoga, 1 August 2025.

Both external checks confirm the same result-stage convention:

* the zero-holder crossed the line first;
* the official result later demoted that runner;
* the promoted winner retains a positive source distance.

All seven populated-comment residuals therefore support the interpretation that `pos` may represent the amended official result while `ovr_btn` and `btn` remain anchored to the physical finishing order.


In [30]:
# Build a compact manual-verification queue for the 11 single-zero-holder
# races whose source comments are entirely blank.
#
# Input grain:
#   One governed source runner row from a single-zero-holder race.
#
# Output grain:
#   One summary row per provisional race requiring external verification.
#
# Purpose:
#   Manual research must use exact source identities rather than relying on
#   memory or shortened notebook prose. This table identifies:
#
#   - the candidate race identity;
#   - the supplied race name;
#   - the later-positioned zero-holder;
#   - the official source winner;
#   - the relevant raw positions and distances; and
#   - the source runner count.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - Raw source values remain unchanged.
#   - `zero_holder_horse` is an observed source role, not yet a confirmed
#     physical-winner classification.
#   - Each external lookup must later be recorded separately in the governed
#     manual-verification register.

# Select the complete bounded population of all-comments-blank residual races.
blank_single_zero_race_keys = (
    single_zero_comment_profile.loc[
        single_zero_comment_profile[
            "comment_evidence_state"
        ].eq("all_comments_blank"),
        ["date", "course", "off"],
    ]
    .drop_duplicates()
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Validate the expected residual population.
if len(blank_single_zero_race_keys) != 11:
    raise ValueError(
        "Expected 11 all-comments-blank residual races, "
        f"but found {len(blank_single_zero_race_keys)}."
    )

# Build a parameterised SQLite condition for the 11 race identities.
blank_single_zero_condition = " OR ".join(
    ["(date = ? AND course = ? AND off = ?)"]
    * len(blank_single_zero_race_keys)
)

blank_single_zero_parameters = [
    value
    for race_key in blank_single_zero_race_keys.itertuples(
        index=False,
        name=None,
    )
    for value in race_key
]

blank_single_zero_context_query = f"""
SELECT
    rowid AS source_rowid,
    date,
    course,
    off,
    race_id,
    race_name,
    horse,
    pos AS raw_pos,
    ovr_btn AS raw_ovr_btn,
    btn AS raw_btn,
    ran AS raw_ran
FROM {SOURCE_TABLE}
WHERE {DATA_ROW_PREDICATE}
  AND ({blank_single_zero_condition})
ORDER BY
    date,
    course,
    off,
    source_rowid
"""

# Retrieve complete source context through the read-only SQLite URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    blank_single_zero_context = pd.read_sql_query(
        blank_single_zero_context_query,
        connection,
        params=blank_single_zero_parameters,
    )

# Create numeric analytical copies without changing the raw columns.
blank_single_zero_context["numeric_pos"] = pd.to_numeric(
    blank_single_zero_context["raw_pos"],
    errors="coerce",
)

blank_single_zero_context["numeric_ovr_btn"] = pd.to_numeric(
    blank_single_zero_context["raw_ovr_btn"],
    errors="coerce",
)

blank_single_zero_context["numeric_btn"] = pd.to_numeric(
    blank_single_zero_context["raw_btn"],
    errors="coerce",
)

# Identify the later-positioned runner carrying zero in both fields.
blank_zero_holders = (
    blank_single_zero_context.loc[
        blank_single_zero_context["numeric_pos"].gt(1)
        & blank_single_zero_context["numeric_ovr_btn"].eq(0)
        & blank_single_zero_context["numeric_btn"].eq(0),
        [
            "date",
            "course",
            "off",
            "horse",
            "raw_pos",
            "raw_ovr_btn",
            "raw_btn",
        ],
    ]
    .rename(
        columns={
            "horse": "zero_holder_horse",
            "raw_pos": "zero_holder_source_pos",
            "raw_ovr_btn": "zero_holder_raw_ovr_btn",
            "raw_btn": "zero_holder_raw_btn",
        }
    )
)

# Identify the official source winner and its positive source distances.
blank_official_winners = (
    blank_single_zero_context.loc[
        blank_single_zero_context["numeric_pos"].eq(1),
        [
            "date",
            "course",
            "off",
            "race_id",
            "race_name",
            "horse",
            "raw_pos",
            "raw_ovr_btn",
            "raw_btn",
            "raw_ran",
        ],
    ]
    .rename(
        columns={
            "horse": "official_winner_horse",
            "raw_pos": "official_winner_source_pos",
            "raw_ovr_btn": "official_winner_raw_ovr_btn",
            "raw_btn": "official_winner_raw_btn",
            "raw_ran": "source_ran",
        }
    )
)

# Combine the two observed source roles at one-row-per-race grain.
blank_single_zero_manual_queue = (
    blank_official_winners
    .merge(
        blank_zero_holders,
        on=["date", "course", "off"],
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

# Confirm that every blank-comment residual race has exactly one official
# winner and one later zero-holder.
if len(blank_single_zero_manual_queue) != 11:
    raise ValueError(
        "Expected 11 complete manual-verification queue rows, "
        f"but found {len(blank_single_zero_manual_queue)}."
    )

display(blank_single_zero_manual_queue)

,date,course,off,race_id,race_name,official_winner_horse,official_winner_source_pos,official_winner_raw_ovr_btn,official_winner_raw_btn,source_ran,zero_holder_horse,zero_holder_source_pos,zero_holder_raw_ovr_btn,zero_holder_raw_btn
0,2017-04-11,Saint-Cloud (FR),12:47,673268,Prix du Pont de Flandre (Handicap) (4yo+) (Turf),Landjunge (GER),1,0.20,0.20,16,Nardo (FR),3,0.0,0.0
1,2018-10-21,Keeneland (USA),9:57,714713,Rood & Riddle Dowager Stakes (3yo+ Fillies & Mares) (Turf),Vexatious (USA),1,0.30,0.30,9,Beach Flower (USA),3,0.0,0.0
2,2019-10-14,Gulfstream Park West (USA),9:09,742692,Maiden Special Weight (Maiden) (2yo Fillies) (Turf),Cheermeister (USA),1,2.00,2.00,12,Lady Panda (USA),2,0.0,0.0
3,2020-04-09,Gulfstream Park (USA),6:30,755633,Claiming Race (Claimer) (3yo+) (Main Track) (Dirt),Congrats This (USA),1,0.30,0.30,7,Red Fog (USA),2,0.0,0.0
4,2020-05-13,Will Rogers Downs (USA),11:45,757124,Maiden Claiming Race (3yo+ Fillies & Mares) (Dirt),Flying Lindy (USA),1,0.30,0.30,9,Theycallherpinky (USA),2,0.0,0.0
5,2020-05-28,Fonner Park (USA),12:42,758096,Claiming Race (3yo+ Fillies & Mares) (Main Track) (Dirt),Our Anabelle (USA),1,0.50,0.50,8,Shimmering Dream (USA),2,0.0,0.0
6,2023-08-31,Longchamp (FR),12:48,848966,Prix de Fontenoy (Maiden) (Unraced 2yo Colts & Geldings) (Grande Course) (Turf),Lord Sinclair (FR),1,1.50,1.50,8,Cabernet Franc (FR),2,0.0,0.0
7,2023-09-08,Saint-Cloud (FR),1:35,849331,Prix Jean-Claude Desaint (Championnat Paris-Turf des Apprentices & Young Jockeys) (Handicap) (4yo+),Toriano (FR),1,0.10,0.10,12,Savile Row (FR),2,0.0,0.0
8,2024-04-20,Randwick (AUS),4:15,866350,Gow-Gates Frank Packer Plate (3yo) (Turf),Kintyre (AUS),1,0.05,0.05,10,Gold Bullion (NZ),2,0.0,0.0
9,2025-08-31,Del Mar (USA),1:39,902849,Green Flash Handicap (3yo+) (Turf),Motorious (GB),1,0.05,0.05,12,Reef Runner (USA),2,0.0,0.0


### Blank-comment single-zero-holder races

All 11 single-zero-holder races with entirely blank source comments were manually verified and captured in the governed manual-verification register.

The checks produced two distinct outcomes.

#### Amended official results

Eight races confirm the established result-stage convention:

* the runner carrying `ovr_btn = 0` and `btn = 0` passed the post first;
* that runner was subsequently disqualified or demoted;
* another runner became the official winner; and
* the promoted official winner retains a positive source distance from the physical winner.

The governed verification identifiers are:

* `NB15-BTN-0008`;
* `NB15-BTN-0009`;
* `NB15-BTN-0011`;
* `NB15-BTN-0012`;
* `NB15-BTN-0014`;
* `NB15-BTN-0015`;
* `NB15-BTN-0016`;
* `NB15-BTN-0017`.

These races provide external confirmation that official `pos` and the finishing stage represented by the distance fields may differ.

#### Source-distance defects

Three races do not contain an identified amended result:

* Saint-Cloud, 11 April 2017;
* Gulfstream Park, 9 April 2020;
* Longchamp, 31 August 2023.

Their published results show ordinary finishing orders incompatible with the later-positioned runner carrying the zero-distance reference.

The governed verification identifiers are:

* `NB15-BTN-0007`;
* `NB15-BTN-0010`;
* `NB15-BTN-0013`.

These cases must be preserved as raw source contradictions and corrected only through a governed downstream reconciliation layer.

### Completed single-zero-holder finding

Across the 322 single-zero-holder races:

* 304 are explained by explicit source-comment evidence;
* five additional populated-comment races were resolved through broader comment review;
* two populated-comment races required external verification;
* eight blank-comment races were externally confirmed as amended results; and
* three blank-comment races were externally confirmed as source-distance defects.

A single later-positioned `0 / 0` runner is therefore highly diagnostic of an amended result, but it is not infallible. Source comments or governed external evidence remain necessary before applying the interpretation to an individual race.


## 4. Zero adjacent margins with positive overall distance

The remaining later-finisher zero population contains 2,750 rows where:

* `ovr_btn` is positive; and
* `btn` is zero.

Unlike the `both_zero` population, these runners do not occupy the race’s zero-distance reference point. They are recorded at a positive overall distance from that reference while carrying no margin from the preceding runner.

Possible explanations include:

* dead heats or tied placings away from first;
* source rounding of very small adjacent margins;
* amended official positions;
* duplicated or irregular finishing positions;
* missing preceding runners; or
* source defects.

The first step tests the relationship between `btn = 0`, repeated official positions and explicit dead-heat wording.


In [31]:
# Profile `btn = 0` among later numeric finishers against repeated official
# positions and source-comment dead-heat evidence.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One summary row per observed structural and comment-evidence state.
#
# Purpose:
#   A zero adjacent margin may represent a tie with the preceding physical
#   finisher, but that interpretation must be tested against the surrounding
#   race structure rather than assumed from `btn = 0` alone.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - Raw `pos`, `ovr_btn`, `btn` and `comment` remain unchanged.
#   - Repeated official positions are measured within the candidate race.
#   - Comment matching is evidence discovery only.
#   - This cell does not yet classify individual rows as dead heats.

btn_zero_structure_query = f"""
WITH governed_rows AS (
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        horse,
        pos AS raw_pos,
        ovr_btn AS raw_ovr_btn,
        btn AS raw_btn,
        comment AS raw_comment
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
),
numeric_position_counts AS (
    SELECT
        date,
        course,
        off,
        raw_pos,
        COUNT(*) AS rows_at_position
    FROM governed_rows
    WHERE typeof(raw_pos) IN ('integer', 'real')
      AND raw_pos > 0
    GROUP BY
        date,
        course,
        off,
        raw_pos
),
btn_zero_rows AS (
    SELECT
        rows.source_rowid,
        rows.date,
        rows.course,
        rows.off,
        rows.horse,
        rows.raw_pos,
        rows.raw_ovr_btn,
        rows.raw_btn,
        rows.raw_comment,
        position_counts.rows_at_position,
        CASE
            WHEN position_counts.rows_at_position > 1
                THEN 'repeated_official_position'
            ELSE 'unique_official_position'
        END AS position_structure,
        CASE
            WHEN LOWER(COALESCE(rows.raw_comment, '')) LIKE '%dead heat%'
              OR LOWER(COALESCE(rows.raw_comment, '')) LIKE '%dead-heat%'
              OR LOWER(COALESCE(rows.raw_comment, '')) LIKE '%deadheated%'
              OR LOWER(COALESCE(rows.raw_comment, '')) LIKE '%dead-heated%'
              OR LOWER(COALESCE(rows.raw_comment, '')) LIKE '%tied%'
              OR LOWER(COALESCE(rows.raw_comment, '')) LIKE '%shared%'
                THEN 'explicit_tie_wording'
            WHEN TRIM(COALESCE(rows.raw_comment, '')) = ''
                THEN 'blank_comment'
            ELSE 'populated_without_tie_wording'
        END AS comment_evidence_state
    FROM governed_rows AS rows
    INNER JOIN numeric_position_counts AS position_counts
        ON rows.date = position_counts.date
       AND rows.course = position_counts.course
       AND rows.off = position_counts.off
       AND rows.raw_pos = position_counts.raw_pos
    WHERE typeof(rows.raw_pos) IN ('integer', 'real')
      AND rows.raw_pos > 1
      AND typeof(rows.raw_ovr_btn) IN ('integer', 'real')
      AND rows.raw_ovr_btn > 0
      AND typeof(rows.raw_btn) IN ('integer', 'real')
      AND rows.raw_btn = 0
)
SELECT
    position_structure,
    comment_evidence_state,
    COUNT(*) AS runner_rows,
    COUNT(DISTINCT date || '|' || course || '|' || off) AS provisional_races,
    MIN(raw_pos) AS minimum_raw_pos,
    MAX(raw_pos) AS maximum_raw_pos,
    MIN(raw_ovr_btn) AS minimum_raw_ovr_btn,
    MAX(raw_ovr_btn) AS maximum_raw_ovr_btn
FROM btn_zero_rows
GROUP BY
    position_structure,
    comment_evidence_state
ORDER BY
    position_structure,
    comment_evidence_state
"""

# Execute against the immutable source through the read-only SQLite URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    btn_zero_structure_profile = pd.read_sql_query(
        btn_zero_structure_query,
        connection,
    )

# Validate that the complete `btn_zero_only` population is represented.
observed_btn_zero_only_rows = int(
    btn_zero_structure_profile["runner_rows"].sum()
)

if observed_btn_zero_only_rows != 2_750:
    raise ValueError(
        "Expected 2,750 btn-zero-only rows, "
        f"but found {observed_btn_zero_only_rows:,}."
    )

display(btn_zero_structure_profile)

,position_structure,comment_evidence_state,runner_rows,provisional_races,minimum_raw_pos,maximum_raw_pos,minimum_raw_ovr_btn,maximum_raw_ovr_btn
0,repeated_official_position,blank_comment,885,852,2,15,0.05,35.50
1,repeated_official_position,explicit_tie_wording,246,246,2,10,0.05,46.00
2,repeated_official_position,populated_without_tie_wording,1571,1560,2,24,0.05,104.50
3,unique_official_position,blank_comment,21,15,4,15,2.00,53.00
4,unique_official_position,explicit_tie_wording,14,14,2,7,0.10,32.75
5,unique_official_position,populated_without_tie_wording,13,12,3,14,0.20,35.00


### Initial structural evidence

The 2,750 later-finisher rows with positive `ovr_btn` and zero `btn` divide sharply by official-position structure:

* 2,702 rows have a repeated official position;
* 48 rows have a unique official position.

Repeated positions therefore account for approximately 98% of the observed population and strongly support the interpretation that zero `btn` usually records no measured separation from another finisher.

Explicit tie wording appears in only a minority of comments. This does not weaken the structural result because many source comments describe running performance without restating the published dead heat.

The 48 unique-position rows are the exceptional population. They may represent:

* amended results where official positions no longer preserve the physical tie;
* incomplete or omitted runner records;
* jurisdiction-specific position conventions;
* rounded or suppressed very small margins; or
* source defects.

Because this population is bounded, it should be reviewed in complete race context before the repeated-position population is treated as the dominant semantic rule.


In [32]:
# Retrieve complete race context for every `btn_zero_only` row whose official
# position is unique within its provisional race.
#
# Input grain:
#   One governed source runner row.
#
# Output grain:
#   One source runner row from a provisional race containing at least one
#   unique-position `btn_zero_only` runner.
#
# Purpose:
#   Repeated official positions explain 2,702 of the 2,750 rows. The remaining
#   48 runner rows are structurally exceptional and require complete race
#   context before any general semantic rule is assigned.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - Raw source values and comments remain unchanged.
#   - The selected 48 rows may occur in fewer than 48 races.
#   - Complete runner context is returned for every affected race.
#   - This cell performs evidence retrieval only; it does not automatically
#     classify the exceptional rows.

unique_btn_zero_query = f"""
WITH governed_rows AS (
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_id,
        race_name,
        type,
        horse,
        pos AS raw_pos,
        ovr_btn AS raw_ovr_btn,
        btn AS raw_btn,
        ran AS raw_ran,
        num AS raw_num,
        comment AS raw_comment
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
),
numeric_position_counts AS (
    SELECT
        date,
        course,
        off,
        raw_pos,
        COUNT(*) AS rows_at_position
    FROM governed_rows
    WHERE typeof(raw_pos) IN ('integer', 'real')
      AND raw_pos > 0
    GROUP BY
        date,
        course,
        off,
        raw_pos
),
selected_races AS (
    SELECT DISTINCT
        rows.date,
        rows.course,
        rows.off
    FROM governed_rows AS rows
    INNER JOIN numeric_position_counts AS position_counts
        ON rows.date = position_counts.date
       AND rows.course = position_counts.course
       AND rows.off = position_counts.off
       AND rows.raw_pos = position_counts.raw_pos
    WHERE typeof(rows.raw_pos) IN ('integer', 'real')
      AND rows.raw_pos > 1
      AND typeof(rows.raw_ovr_btn) IN ('integer', 'real')
      AND rows.raw_ovr_btn > 0
      AND typeof(rows.raw_btn) IN ('integer', 'real')
      AND rows.raw_btn = 0
      AND position_counts.rows_at_position = 1
)
SELECT
    rows.source_rowid,
    rows.date,
    rows.course,
    rows.off,
    rows.race_id,
    rows.race_name,
    rows.type,
    rows.horse,
    rows.raw_pos,
    rows.raw_ovr_btn,
    rows.raw_btn,
    rows.raw_ran,
    rows.raw_num,
    rows.raw_comment,
    position_counts.rows_at_position,
    CASE
        WHEN typeof(rows.raw_pos) IN ('integer', 'real')
         AND rows.raw_pos > 1
         AND typeof(rows.raw_ovr_btn) IN ('integer', 'real')
         AND rows.raw_ovr_btn > 0
         AND typeof(rows.raw_btn) IN ('integer', 'real')
         AND rows.raw_btn = 0
         AND position_counts.rows_at_position = 1
            THEN 1
        ELSE 0
    END AS selected_unique_btn_zero_row
FROM governed_rows AS rows
INNER JOIN selected_races
    ON rows.date = selected_races.date
   AND rows.course = selected_races.course
   AND rows.off = selected_races.off
LEFT JOIN numeric_position_counts AS position_counts
    ON rows.date = position_counts.date
   AND rows.course = position_counts.course
   AND rows.off = position_counts.off
   AND rows.raw_pos = position_counts.raw_pos
ORDER BY
    rows.date,
    rows.course,
    rows.off,
    rows.source_rowid
"""

# Execute against the immutable source through the read-only SQLite URI.
with sqlite3.connect(SOURCE_DATABASE_URI, uri=True) as connection:
    unique_btn_zero_context = pd.read_sql_query(
        unique_btn_zero_query,
        connection,
    )

# Validate that the complete exceptional runner population is represented.
selected_unique_btn_zero_rows = int(
    unique_btn_zero_context["selected_unique_btn_zero_row"].sum()
)

if selected_unique_btn_zero_rows != 48:
    raise ValueError(
        "Expected 48 unique-position btn-zero-only runner rows, "
        f"but found {selected_unique_btn_zero_rows:,}."
    )

# Produce a compact race-level inventory before displaying complete races.
unique_btn_zero_race_inventory = (
    unique_btn_zero_context
    .groupby(
        ["date", "course", "off"],
        as_index=False,
        dropna=False,
    )
    .agg(
        race_name=("race_name", "first"),
        runner_rows=("source_rowid", "size"),
        selected_unique_btn_zero_rows=(
            "selected_unique_btn_zero_row",
            "sum",
        ),
    )
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

display(unique_btn_zero_race_inventory)

# Display each complete affected race independently. The marker column shows
# exactly which runner row belongs to the exceptional 48-row population.
for race_key, race_rows in unique_btn_zero_context.groupby(
    ["date", "course", "off"],
    sort=True,
    dropna=False,
):
    race_date, race_course, race_off = race_key

    print(f"\n{race_date} — {race_course} — {race_off}")

    display(
        race_rows[
            [
                "source_rowid",
                "race_name",
                "horse",
                "raw_pos",
                "raw_ovr_btn",
                "raw_btn",
                "rows_at_position",
                "selected_unique_btn_zero_row",
                "raw_ran",
                "raw_num",
                "raw_comment",
            ]
        ].reset_index(drop=True)
    )

,date,course,off,race_name,runner_rows,selected_unique_btn_zero_rows
0,2015-05-03,Longchamp (FR),2:40,Prix Vanteaux (3yo Fillies) (Turf),5,1
1,2015-05-16,Auteuil (FR),4:50,Prix RMC (Chase) (Handicap) (5yo+) (Turf),16,1
2,2015-05-16,Morphettville (AUS),4:38,William Hill SA Fillies Classic (3yo Fillies) (Turf),14,1
3,2015-05-27,Longchamp (FR),12:50,Prix du Palais Bourbon (Handicap) (4yo+ Fillies & Mares) (Turf),15,1
4,2015-07-29,Avenches (SWI),4:45,Grand Prix DAvenches (),10,4
5,2015-08-13,Lingfield (AW),7:45,Hickstead Show Ground Handicap,8,1
6,2016-10-23,Wincanton,4:40,Racing UK HD Handicap Hurdle,6,1
7,2016-12-26,Down Royal (IRE),2:30,Enda Bell Maiden Hurdle,16,1
8,2017-08-05,Bad Doberan (GER),1:10,Pastorius - Goldene Peitshce von Bad Doberan (Turf),5,1
9,2018-01-14,Southwell (AW),3:55,sunbets.co.uk Download The App Handicap,10,1



2015-05-03 — Longchamp (FR) — 2:40


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,rows_at_position,selected_unique_btn_zero_row,raw_ran,raw_num,raw_comment
0,48759,Prix Vanteaux (3yo Fillies) (Turf),Burma Sea (FR),5,2.5,0,1.0,1,5,2,Held up in touch in last - pushed along 2f out - not quicken - plugged on under mostly hand ride final furlong and never able to challenge
1,48760,Prix Vanteaux (3yo Fillies) (Turf),Akatea (IRE),4,2.5,1,1.0,0,5,5,Took keen hold - in touch on inner - angled out and shaken up to challenge 2f out - joined leader over 1f out - ridden final furlong - kept on and still every chance until no extra towards finish
2,48761,Prix Vanteaux (3yo Fillies) (Turf),Via Manzoni (IRE),3,1.5,1,1.0,0,5,3,Soon led - headed briefly after 1f - shaken up when strongly pressed 2f out - ridden and joined over 1f out - edged ahead again final furlong - kept on until headed close home - no extra and dropped to 3rd
3,48762,Prix Vanteaux (3yo Fillies) (Turf),Vedouma (FR),2,0.5,0.5,1.0,0,5,1,Held up in touch - ridden to challenge over 1f out - stayed on and every chance final furlong - not quite pace of winner close home
4,48763,Prix Vanteaux (3yo Fillies) (Turf),Olorda (GER),1,0,0,1.0,0,5,4,Led briefly after 1f - otherwise tracked leader - ridden over 2f out - slightly outpaced in 4th entering final furlong - rallied under pressure and stayed on well to challenge final 120yds - got up to lead close home



2015-05-16 — Auteuil (FR) — 4:50


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,rows_at_position,selected_unique_btn_zero_row,raw_ran,raw_num,raw_comment
0,55967,Prix RMC (Chase) (Handicap) (5yo+) (Turf),Ysawa (FR),1,0,0,1.0,0,16,2,
1,55968,Prix RMC (Chase) (Handicap) (5yo+) (Turf),Fafintadenient (GB),2,1.5,1.5,1.0,0,16,4,
2,55969,Prix RMC (Chase) (Handicap) (5yo+) (Turf),Sparkie (FR),3,9.5,8,1.0,0,16,8,
3,55970,Prix RMC (Chase) (Handicap) (5yo+) (Turf),Lamego (FR),6,15.5,6,1.0,0,16,1,Finished dead-heat 4th - disqualified and placed 6th
4,55971,Prix RMC (Chase) (Handicap) (5yo+) (Turf),Valtor (FR),4,15.5,0,1.0,1,16,3,Finished dead-heat 4th - awarded 4th outright
5,55972,Prix RMC (Chase) (Handicap) (5yo+) (Turf),Pull Marine (FR),5,16.5,1,1.0,0,16,15,Finished 6th - placed 5thb
6,55973,Prix RMC (Chase) (Handicap) (5yo+) (Turf),Touareg DAiry (FR),7,17.5,1,1.0,0,16,17,
7,55974,Prix RMC (Chase) (Handicap) (5yo+) (Turf),Fumseck (FR),8,19,1.5,1.0,0,16,13,
8,55975,Prix RMC (Chase) (Handicap) (5yo+) (Turf),Honglong (FR),9,24,5,1.0,0,16,16,
9,55976,Prix RMC (Chase) (Handicap) (5yo+) (Turf),Vado Via (FR),10,24,0.1,1.0,0,16,14,


...

,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,rows_at_position,selected_unique_btn_zero_row,raw_ran,raw_num,raw_comment
0,1795247,Pegasus World Cup Filly & Mare Turf Invitational Stakes (Turf),Destino dOro (USA),1,0,0,1.0,0,12,3,In rear - ridden and headway on outer over 1f out - led inside final 110yds - kept on
1,1795248,Pegasus World Cup Filly & Mare Turf Invitational Stakes (Turf),Crevalle dOro (USA),2,0.5,0.5,1.0,0,12,9,Midfield - headway when hung left 1f out - went second towards finish - kept on
2,1795249,Pegasus World Cup Filly & Mare Turf Invitational Stakes (Turf),Movin On Up (USA),3,1,0.5,1.0,0,12,12,Prominent - led 1f out - headed inside final 110yds - soon no extra and lost second
3,1795250,Pegasus World Cup Filly & Mare Turf Invitational Stakes (Turf),Whiskey Decision (USA),4,1.5,0.5,1.0,0,12,7,Towards rear - going okay but plenty to do under 2f out - switched right and headway 1f out - kept on well final 110yds - eyecatcher
4,1795251,Pegasus World Cup Filly & Mare Turf Invitational Stakes (Turf),Heredia (GB),5,2.25,0.75,1.0,0,12,2,In rear - headway inside final furlong - on outer when kept on final 110yds - never dangerous
5,1795252,Pegasus World Cup Filly & Mare Turf Invitational Stakes (Turf),Caitlinhergrtness (CAN),6,4.5,2.25,1.0,0,12,6,In touch with leaders - weakened final 110yds
6,1795253,Pegasus World Cup Filly & Mare Turf Invitational Stakes (Turf),And One More Time (USA),7,4.75,0.3,1.0,0,12,10,Awkward start - in touch with leaders - headway on outer over 4f out - soon led - headed over 3f out - weakened inside final furlong
7,1795254,Pegasus World Cup Filly & Mare Turf Invitational Stakes (Turf),Proctor Street (USA),8,5.25,0.5,1.0,0,12,8,Midfield - against rail when not clear run over 2f out - soon waiting for room - in the clear and headway over 1f out - weakened final 110yds
8,1795255,Pegasus World Cup Filly & Mare Turf Invitational Stakes (Turf),In Our Time (USA),9,6.5,1.25,1.0,0,12,11,Soon led - headed over 4f out - led again over 3f out - headed 1f out - soon weakened
9,1795256,Pegasus World Cup Filly & Mare Turf Invitational Stakes (Turf),Breath Away (GB),10,6.5,0,1.0,1,12,4,In touch with leaders - weakened inside final furlong



2026-02-14 — Sha Tin — 10:00


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,rows_at_position,selected_unique_btn_zero_row,raw_ran,raw_num,raw_comment
0,1803254,Tvb Miss Hong Kong Pageant Hcp (C3) (Handicap) (Turf),Stormy Grove (AUS),1,0,0,1.0,0,14,8,In rear - switched left over 2f out - good headway from over 1f out - ran on well inside final furlong - led towards finish
1,1803255,Tvb Miss Hong Kong Pageant Hcp (C3) (Handicap) (Turf),Without Compare (AUS),2,0.75,0.75,1.0,0,14,11,Led - ran on from 2f out - reduced lead inside final furlong - headed towards finish
2,1803256,Tvb Miss Hong Kong Pageant Hcp (C3) (Handicap) (Turf),Endued (AUS),3,2.25,1.5,1.0,0,14,10,Midfield early - in touch with leaders after 3f - some headway from over 1f out - kept on inside final furlong
3,1803257,Tvb Miss Hong Kong Pageant Hcp (C3) (Handicap) (Turf),Lucky Sam Gor (NZ),4,2.75,0.5,1.0,0,14,3,Midfield - some headway then waiting for room from 2f out - lost position over 1f out - ran on well inside final furlong
4,1803258,Tvb Miss Hong Kong Pageant Hcp (C3) (Handicap) (Turf),Natural Numbers (AUS),5,3.5,0.75,1.0,0,14,4,In touch with leaders - no impression but kept on from over 1f out
5,1803259,Tvb Miss Hong Kong Pageant Hcp (C3) (Handicap) (Turf),Mighty Masts (AUS),6,3.75,0.25,1.0,0,14,5,Prominent - weakened inside final furlong
6,1803260,Tvb Miss Hong Kong Pageant Hcp (C3) (Handicap) (Turf),Enthusium (GB),7,4.5,0.75,1.0,0,14,7,Towards rear - some headway then not clear run over 1f out - ran on inside final furlong
7,1803261,Tvb Miss Hong Kong Pageant Hcp (C3) (Handicap) (Turf),Chill Easy (GB),8,5.25,0.75,1.0,0,14,6,Midfield - headway over 1f out - weakened inside final furlong
8,1803262,Tvb Miss Hong Kong Pageant Hcp (C3) (Handicap) (Turf),Danicas Choice (NZ),9,5.5,0.25,1.0,0,14,14,Took keen hold - held up in rear - some headway inside final furlong
9,1803263,Tvb Miss Hong Kong Pageant Hcp (C3) (Handicap) (Turf),Glittering Legend (GB),10,7,1.5,1.0,0,14,1,Took keen hold - held up in midfield - towards rear over 2f out - no impression



2026-05-11 — Southwell — 16:12


,source_rowid,race_name,horse,raw_pos,raw_ovr_btn,raw_btn,rows_at_position,selected_unique_btn_zero_row,raw_ran,raw_num,raw_comment
0,1842750,Summer Fun & Makers Market Raceday 27th July Mares Open Maiden NH Flat Race (Cat 3) (GBB) (Div II),Getmyfriend (IRE),1,0,0,1.0,0,10,2,Prominent - ridden and switched left over 2f out - led over 1f out - comfortably(op 10/3 tchd 11/4)
1,1842751,Summer Fun & Makers Market Raceday 27th July Mares Open Maiden NH Flat Race (Cat 3) (GBB) (Div II),Agreymiracleseeker (FR),3,2.75,2.75,1.0,0,10,5,In rear - headway over 2f out - hung left and hampered rival and went second over 1f out - hung left inside final furlong - no extra inside final 110yds - dead-heated for 2nd - placed 3rd outright(op 11/1 tchd 10/1)
2,1842752,Summer Fun & Makers Market Raceday 27th July Mares Open Maiden NH Flat Race (Cat 3) (GBB) (Div II),Crystalate (IRE),2,2.75,0,1.0,1,10,7,Towards rear - headway over 3f out - bumped over 1f out - carried left inside final furlong - kept on and dead-heated for 2nd - placed 2nd outright(op 13/8 tchd 11/8)
3,1842753,Summer Fun & Makers Market Raceday 27th July Mares Open Maiden NH Flat Race (Cat 3) (GBB) (Div II),Golden Bouquet (GB),4,6.25,3.5,1.0,0,10,9,Took keen hold - prominent - led over 2f out - headed over 1f out - slightly hampered and weakened inside final furlong(op 11/4)
4,1842754,Summer Fun & Makers Market Raceday 27th July Mares Open Maiden NH Flat Race (Cat 3) (GBB) (Div II),Cool Million (IRE),5,9.25,3,1.0,0,10,6,Midfield - headway over 3f out - hampered and bumped rival over 1f out - weakened inside final furlong(op 16/1 tchd 14/1)
5,1842755,Summer Fun & Makers Market Raceday 27th July Mares Open Maiden NH Flat Race (Cat 3) (GBB) (Div II),Esther Spring (FR),6,9.5,0.3,1.0,0,10,8,Took keen hold - raced in last - headway and switched right over 2f out - weakened inside final furlong(op 100/1)
6,1842756,Summer Fun & Makers Market Raceday 27th July Mares Open Maiden NH Flat Race (Cat 3) (GBB) (Div II),Penny Rocket (IRE),7,18,8.5,1.0,0,10,10,Midfield - outpaced and dropped to rear over 2f out - weakened over 1f out(op 10/1 tchd 17/2)
7,1842757,Summer Fun & Makers Market Raceday 27th July Mares Open Maiden NH Flat Race (Cat 3) (GBB) (Div II),Little Mayhem (GB),8,23,5,1.0,0,10,3,In rear - weakened over 2f out(op 100/1)
8,1842758,Summer Fun & Makers Market Raceday 27th July Mares Open Maiden NH Flat Race (Cat 3) (GBB) (Div II),Roccobye (GB),9,28,5,1.0,0,10,4,Led but pestered - headed and weakened over 2f out(tchd 150/1)
9,1842759,Summer Fun & Makers Market Raceday 27th July Mares Open Maiden NH Flat Race (Cat 3) (GBB) (Div II),Boat That Rocked (IRE),10,33.5,5.5,1.0,0,10,1,Took keen hold - pressed leader - weakened over 2f out(tchd 80/1)


### Unique official positions do not necessarily mean isolated distances

Complete context shows that many unique-position `btn = 0` rows still share their `ovr_btn` value with another runner in the race.

Two mechanisms are visible:

1. **Amended physical ties**

   Two runners finished together physically, but a disqualification or placing amendment separated their official positions. The zero adjacent margin continues to describe the physical tie even though `pos` is unique.

2. **Distinct official positions with equal recorded cumulative distance**

   Some source results assign consecutive or nearby official positions while recording the same `ovr_btn` for several runners. The later runner then receives `btn = 0`.

The second structure may reflect jurisdiction-specific result presentation, distance rounding, capped or grouped margins, or source defects. It should not automatically be interpreted as an official dead heat.

The next test measures whether each exceptional row has another same-race runner with the identical recorded `ovr_btn`, and describes that counterpart’s result structure.


In [33]:
# Profile same-race `ovr_btn` counterparts for the 48 unique-position rows
# carrying positive `ovr_btn` and zero `btn`.
#
# Input grain:
#   One governed source runner row in a race containing an exceptional
#   unique-position `btn_zero_only` row.
#
# Output grain:
#   One analytical row per selected exceptional runner, followed by a summary
#   of its observed same-distance counterpart structure.
#
# Purpose:
#   Complete review suggests that these rows are not isolated zero margins.
#   They generally share their cumulative distance with another runner even
#   where official positions differ. This cell tests that proposition across
#   the complete 48-row population.
#
# Important:
#   - Candidate race identity remains `date + course + off`.
#   - Raw source columns remain unchanged.
#   - Distance equality is exact equality in the stored numeric value.
#   - A matching nonnumeric position such as `DSQ` can represent the other
#     member of a physical tie removed from the numeric official sequence.
#   - Exact distance equality does not by itself prove an official dead heat.

# Work from the complete race context already retrieved.
unique_btn_zero_analysis = unique_btn_zero_context.copy()

# Create numeric analytical copies while preserving all raw columns.
unique_btn_zero_analysis["numeric_pos"] = pd.to_numeric(
    unique_btn_zero_analysis["raw_pos"],
    errors="coerce",
)

unique_btn_zero_analysis["numeric_ovr_btn"] = pd.to_numeric(
    unique_btn_zero_analysis["raw_ovr_btn"],
    errors="coerce",
)

# Retain the 48 structurally exceptional runner rows.
selected_unique_btn_zero = (
    unique_btn_zero_analysis.loc[
        unique_btn_zero_analysis[
            "selected_unique_btn_zero_row"
        ].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)

# Self-join each selected row to every other runner in the same race carrying
# the identical stored overall distance.
same_distance_counterparts = (
    selected_unique_btn_zero[
        [
            "source_rowid",
            "date",
            "course",
            "off",
            "horse",
            "raw_pos",
            "numeric_pos",
            "numeric_ovr_btn",
        ]
    ]
    .rename(
        columns={
            "source_rowid": "selected_source_rowid",
            "horse": "selected_horse",
            "raw_pos": "selected_raw_pos",
            "numeric_pos": "selected_numeric_pos",
        }
    )
    .merge(
        unique_btn_zero_analysis[
            [
                "source_rowid",
                "date",
                "course",
                "off",
                "horse",
                "raw_pos",
                "numeric_pos",
                "numeric_ovr_btn",
                "raw_btn",
                "raw_comment",
            ]
        ].rename(
            columns={
                "source_rowid": "counterpart_source_rowid",
                "horse": "counterpart_horse",
                "raw_pos": "counterpart_raw_pos",
                "numeric_pos": "counterpart_numeric_pos",
                "raw_btn": "counterpart_raw_btn",
                "raw_comment": "counterpart_raw_comment",
            }
        ),
        on=[
            "date",
            "course",
            "off",
            "numeric_ovr_btn",
        ],
        how="left",
    )
)

# Exclude the selected row matching itself.
same_distance_counterparts = same_distance_counterparts.loc[
    same_distance_counterparts["selected_source_rowid"].ne(
        same_distance_counterparts["counterpart_source_rowid"]
    )
].copy()

# Count and classify counterpart positions for each selected runner.
same_distance_counterpart_profile = (
    same_distance_counterparts
    .groupby(
        [
            "selected_source_rowid",
            "date",
            "course",
            "off",
            "selected_horse",
            "selected_raw_pos",
            "numeric_ovr_btn",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        same_distance_counterpart_rows=(
            "counterpart_source_rowid",
            "size",
        ),
        numeric_counterpart_rows=(
            "counterpart_numeric_pos",
            lambda values: int(values.notna().sum()),
        ),
        nonnumeric_counterpart_rows=(
            "counterpart_numeric_pos",
            lambda values: int(values.isna().sum()),
        ),
        minimum_numeric_counterpart_pos=(
            "counterpart_numeric_pos",
            "min",
        ),
        maximum_numeric_counterpart_pos=(
            "counterpart_numeric_pos",
            "max",
        ),
        counterpart_horses=(
            "counterpart_horse",
            lambda values: " | ".join(
                values.dropna().astype(str).tolist()
            ),
        ),
        counterpart_positions=(
            "counterpart_raw_pos",
            lambda values: " | ".join(
                values.dropna().astype(str).tolist()
            ),
        ),
    )
)

# Reattach any selected row with no same-distance counterpart so absence would
# remain explicit rather than being silently dropped.
selected_counterpart_profile = (
    selected_unique_btn_zero[
        [
            "source_rowid",
            "date",
            "course",
            "off",
            "horse",
            "raw_pos",
            "numeric_ovr_btn",
        ]
    ]
    .rename(
        columns={
            "source_rowid": "selected_source_rowid",
            "horse": "selected_horse",
            "raw_pos": "selected_raw_pos",
        }
    )
    .merge(
        same_distance_counterpart_profile,
        on=[
            "selected_source_rowid",
            "date",
            "course",
            "off",
            "selected_horse",
            "selected_raw_pos",
            "numeric_ovr_btn",
        ],
        how="left",
        validate="one_to_one",
    )
)

# Assign a structural state without claiming the sporting reason.
selected_counterpart_profile["counterpart_structure"] = (
    "same_distance_counterpart_present"
)

selected_counterpart_profile.loc[
    selected_counterpart_profile[
        "same_distance_counterpart_rows"
    ].isna(),
    "counterpart_structure",
] = "no_same_distance_counterpart"

selected_counterpart_profile.loc[
    selected_counterpart_profile[
        "nonnumeric_counterpart_rows"
    ].fillna(0).gt(0),
    "counterpart_structure",
] = "includes_nonnumeric_counterpart"

selected_counterpart_profile.loc[
    selected_counterpart_profile[
        "same_distance_counterpart_rows"
    ].fillna(0).gt(1),
    "counterpart_structure",
] = "multiple_same_distance_counterparts"

# Validate the complete exceptional population.
if len(selected_counterpart_profile) != 48:
    raise ValueError(
        "Expected 48 selected counterpart-profile rows, "
        f"but found {len(selected_counterpart_profile)}."
    )

# Summarise the observed structures.
selected_counterpart_summary = (
    selected_counterpart_profile
    .groupby(
        "counterpart_structure",
        as_index=False,
        dropna=False,
    )
    .agg(
        runner_rows=("selected_source_rowid", "size"),
        provisional_races=(
            "date",
            lambda values: len(
                set(
                    zip(
                        values,
                        selected_counterpart_profile.loc[
                            values.index,
                            "course",
                        ],
                        selected_counterpart_profile.loc[
                            values.index,
                            "off",
                        ],
                    )
                )
            ),
        ),
        minimum_same_distance_counterparts=(
            "same_distance_counterpart_rows",
            "min",
        ),
        maximum_same_distance_counterparts=(
            "same_distance_counterpart_rows",
            "max",
        ),
    )
    .sort_values(
        "runner_rows",
        ascending=False,
        kind="stable",
    )
    .reset_index(drop=True)
)

display(selected_counterpart_summary)

# Display any row with no exact same-distance counterpart because such a row
# would require a different explanation for its zero adjacent margin.
display(
    selected_counterpart_profile.loc[
        selected_counterpart_profile[
            "counterpart_structure"
        ].eq("no_same_distance_counterpart")
    ].reset_index(drop=True)
)

,counterpart_structure,runner_rows,provisional_races,minimum_same_distance_counterparts,maximum_same_distance_counterparts
0,same_distance_counterpart_present,31,31,1,1
1,multiple_same_distance_counterparts,13,5,2,4
2,includes_nonnumeric_counterpart,4,4,1,1


,selected_source_rowid,date,course,off,selected_horse,selected_raw_pos,numeric_ovr_btn,same_distance_counterpart_rows,numeric_counterpart_rows,nonnumeric_counterpart_rows,minimum_numeric_counterpart_pos,maximum_numeric_counterpart_pos,counterpart_horses,counterpart_positions,counterpart_structure


### Same-distance counterpart result

All 48 unique-position `btn = 0` rows have at least one other runner in the same race with the identical stored `ovr_btn`.

The observed structures are:

* 31 rows with one numeric same-distance counterpart;
* 13 rows with multiple same-distance counterparts;
* four rows with a nonnumeric same-distance counterpart such as `DSQ`.

No selected row lacks an exact same-distance counterpart.

This establishes that `btn = 0` is not an isolated missing value in this population. It records membership of a same-distance group, even where the official `pos` values no longer show a repeated placing.

The next step determines how those same-distance counterparts relate to the selected runner in the official result order.


In [34]:
# Classify the official-position relationship between each of the 48 selected
# unique-position `btn = 0` runners and its same-distance counterparts.
#
# Input grain:
#   One selected exceptional runner joined to one same-distance counterpart.
#
# Output grain:
#   One summary row per observed counterpart-position relationship.
#
# Purpose:
#   We have established that every exceptional zero adjacent margin belongs
#   to an exact same-distance group. This cell determines whether the matching
#   runner is adjacent in the official result, separated by amended placings,
#   nonnumeric, or part of a larger grouped-distance tail.
#
# Important:
#   - Raw source values remain unchanged.
#   - Numeric position differences describe official source positions only.
#   - A difference of -1 means the counterpart is one official place ahead.
#   - A difference of +1 means the counterpart is one official place behind.
#   - Nonnumeric counterparts remain a separate structural category.
#   - These structures are descriptive and do not themselves prove why the
#     distance was shared.

# Start from the same-distance counterpart join already produced.
counterpart_relationships = same_distance_counterparts.copy()

# Derive the official numeric position difference where both positions are
# numeric. Positive means the counterpart is officially behind the selected
# runner; negative means it is officially ahead.
counterpart_relationships["official_position_difference"] = (
    counterpart_relationships["counterpart_numeric_pos"]
    - counterpart_relationships["selected_numeric_pos"]
)

# Assign one descriptive relationship per selected-counterpart pair.
counterpart_relationships["position_relationship"] = (
    "nonadjacent_numeric_counterpart"
)

counterpart_relationships.loc[
    counterpart_relationships[
        "counterpart_numeric_pos"
    ].isna(),
    "position_relationship",
] = "nonnumeric_counterpart"

counterpart_relationships.loc[
    counterpart_relationships[
        "official_position_difference"
    ].eq(-1),
    "position_relationship",
] = "immediately_preceding_official_position"

counterpart_relationships.loc[
    counterpart_relationships[
        "official_position_difference"
    ].eq(1),
    "position_relationship",
] = "immediately_following_official_position"

counterpart_relationships.loc[
    counterpart_relationships[
        "official_position_difference"
    ].eq(0),
    "position_relationship",
] = "same_official_position"

# Summarise counterpart-pair relationships.
counterpart_relationship_summary = (
    counterpart_relationships
    .groupby(
        "position_relationship",
        as_index=False,
        dropna=False,
    )
    .agg(
        counterpart_pairs=("counterpart_source_rowid", "size"),
        selected_runner_rows=(
            "selected_source_rowid",
            "nunique",
        ),
        provisional_races=(
            "date",
            lambda values: len(
                set(
                    zip(
                        values,
                        counterpart_relationships.loc[
                            values.index,
                            "course",
                        ],
                        counterpart_relationships.loc[
                            values.index,
                            "off",
                        ],
                    )
                )
            ),
        ),
        minimum_position_difference=(
            "official_position_difference",
            "min",
        ),
        maximum_position_difference=(
            "official_position_difference",
            "max",
        ),
    )
    .sort_values(
        "counterpart_pairs",
        ascending=False,
        kind="stable",
    )
    .reset_index(drop=True)
)

display(counterpart_relationship_summary)

# Build one row per selected runner showing all observed counterpart positions.
selected_relationship_profile = (
    counterpart_relationships
    .groupby(
        [
            "selected_source_rowid",
            "date",
            "course",
            "off",
            "selected_horse",
            "selected_raw_pos",
            "numeric_ovr_btn",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        counterpart_count=("counterpart_source_rowid", "size"),
        counterpart_positions=(
            "counterpart_raw_pos",
            lambda values: " | ".join(
                values.dropna().astype(str).tolist()
            ),
        ),
        position_relationships=(
            "position_relationship",
            lambda values: " | ".join(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
    )
)

# Display rows whose same-distance counterpart is neither immediately adjacent
# nor nonnumeric. These represent larger or irregular same-distance groups.
display(
    selected_relationship_profile.loc[
        selected_relationship_profile[
            "position_relationships"
        ].str.contains(
            "nonadjacent_numeric_counterpart",
            na=False,
        )
    ]
    .sort_values(
        ["date", "course", "off", "selected_raw_pos"],
        kind="stable",
    )
    .reset_index(drop=True)
)

,position_relationship,counterpart_pairs,selected_runner_rows,provisional_races,minimum_position_difference,maximum_position_difference
0,immediately_preceding_official_position,38,38,31,-1.0,-1.0
1,nonadjacent_numeric_counterpart,24,15,7,-5.0,3.0
2,immediately_following_official_position,9,9,5,1.0,1.0
3,nonnumeric_counterpart,4,4,4,NaN,NaN


,selected_source_rowid,date,course,off,selected_horse,selected_raw_pos,numeric_ovr_btn,counterpart_count,counterpart_positions,position_relationships
0,55971,2015-05-16,Auteuil (FR),4:50,Valtor (FR),4,15.50,1,6,nonadjacent_numeric_counterpart
1,55517,2015-05-16,Morphettville (AUS),4:38,Mighty Maher (AUS),12,8.75,1,10,nonadjacent_numeric_counterpart
2,90272,2015-07-29,Avenches (SWI),4:45,Fordson (IRE),7,7.25,4,10 | 9 | 8 | 6,immediately_following_official_position | immediately_preceding_official_position | nonadjacent_numeric_counterpart
3,90271,2015-07-29,Avenches (SWI),4:45,Bantu (FR),8,7.25,4,10 | 9 | 7 | 6,immediately_following_official_position | immediately_preceding_official_position | nonadjacent_numeric_counterpart
4,90270,2015-07-29,Avenches (SWI),4:45,Theodore Gericault (IRE),9,7.25,4,10 | 8 | 7 | 6,immediately_following_official_position | immediately_preceding_official_position | nonadjacent_numeric_counterpart
5,90269,2015-07-29,Avenches (SWI),4:45,Fabrino (IRE),10,7.25,4,9 | 8 | 7 | 6,immediately_preceding_official_position | nonadjacent_numeric_counterpart
6,570605,2018-07-14,Delaware Park (USA),9:46,Archaggelos (USA),6,3.25,2,5 | 4,immediately_preceding_official_position | nonadjacent_numeric_counterpart
7,875197,2020-08-15,Merano (ITY),5:55,Quita (GB),7,8.25,2,4 | 9,nonadjacent_numeric_counterpart
8,875198,2020-08-15,Merano (ITY),5:55,Volkovka (FR),9,8.25,2,4 | 7,nonadjacent_numeric_counterpart
9,1632895,2025-02-09,St Moritz (SWI),12:30,Gordon Grey (IRE),7,35.00,3,9 | 8 | 6,immediately_following_official_position | immediately_preceding_official_position | nonadjacent_numeric_counterpart


### Official-position relationships within same-distance groups

Every unique-position `btn = 0` runner belongs to a same-distance group.

For most selected runners, at least one same-distance counterpart occupies the immediately preceding official position. This is consistent with `btn` representing the distance from the preceding physical finisher or distance group.

The apparent nonadjacent relationships largely arise because:

* a same-distance group contains three or more runners;
* a tied runner was disqualified or demoted away from its physical finishing position; or
* several trailing runners share one capped or grouped cumulative distance.

Pair-level counts therefore overstate the number of exceptional mechanisms. Each selected runner must instead be assigned one mutually exclusive structural classification.


In [35]:
# Assign one mutually exclusive same-distance structure to each of the 48
# unique-position runners with positive `ovr_btn` and zero `btn`.
#
# Input grain:
#   One selected exceptional runner with all same-distance counterparts.
#
# Output grain:
#   One analytical row per selected runner, followed by a summary of the
#   mutually exclusive structures.
#
# Purpose:
#   Pair-level relationship counts include several counterparts for runners
#   belonging to larger same-distance groups. This cell reduces the evidence
#   to one structural classification per selected runner.
#
# Classification precedence:
#   1. A nonnumeric counterpart indicates that a matching runner has been
#      removed from the numeric official sequence, commonly through DSQ.
#   2. More than one counterpart indicates a multi-runner distance plateau.
#   3. A sole immediately preceding counterpart represents an adjacent
#      same-distance pair.
#   4. Any remaining sole numeric counterpart is a separated same-distance
#      pair whose official positions are nonadjacent.
#
# Important:
#   - These are structural descriptions, not final sporting interpretations.
#   - Raw source values remain unchanged.
#   - Comment evidence is attached separately and does not override structure.

# Aggregate the pair-level relationships to one row per selected runner.
selected_distance_group_structure = (
    counterpart_relationships
    .groupby(
        [
            "selected_source_rowid",
            "date",
            "course",
            "off",
            "selected_horse",
            "selected_raw_pos",
            "numeric_ovr_btn",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        counterpart_count=(
            "counterpart_source_rowid",
            "size",
        ),
        numeric_counterpart_count=(
            "counterpart_numeric_pos",
            lambda values: int(values.notna().sum()),
        ),
        nonnumeric_counterpart_count=(
            "counterpart_numeric_pos",
            lambda values: int(values.isna().sum()),
        ),
        has_immediately_preceding_counterpart=(
            "official_position_difference",
            lambda values: bool(values.eq(-1).any()),
        ),
        has_immediately_following_counterpart=(
            "official_position_difference",
            lambda values: bool(values.eq(1).any()),
        ),
        minimum_position_difference=(
            "official_position_difference",
            "min",
        ),
        maximum_position_difference=(
            "official_position_difference",
            "max",
        ),
        counterpart_horses=(
            "counterpart_horse",
            lambda values: " | ".join(
                values.dropna().astype(str).tolist()
            ),
        ),
        counterpart_positions=(
            "counterpart_raw_pos",
            lambda values: " | ".join(
                values.dropna().astype(str).tolist()
            ),
        ),
    )
)

# Assign exactly one structural category using explicit precedence.
selected_distance_group_structure["distance_group_structure"] = (
    "separated_same_distance_pair"
)

selected_distance_group_structure.loc[
    selected_distance_group_structure[
        "has_immediately_preceding_counterpart"
    ]
    & selected_distance_group_structure[
        "counterpart_count"
    ].eq(1),
    "distance_group_structure",
] = "adjacent_same_distance_pair"

selected_distance_group_structure.loc[
    selected_distance_group_structure[
        "counterpart_count"
    ].gt(1),
    "distance_group_structure",
] = "multi_runner_distance_plateau"

selected_distance_group_structure.loc[
    selected_distance_group_structure[
        "nonnumeric_counterpart_count"
    ].gt(0),
    "distance_group_structure",
] = "includes_nonnumeric_counterpart"

# Attach the selected runner's source comment for bounded evidence review.
selected_runner_comments = (
    selected_unique_btn_zero[
        [
            "source_rowid",
            "raw_comment",
        ]
    ]
    .rename(
        columns={
            "source_rowid": "selected_source_rowid",
            "raw_comment": "selected_raw_comment",
        }
    )
)

selected_distance_group_structure = (
    selected_distance_group_structure
    .merge(
        selected_runner_comments,
        on="selected_source_rowid",
        how="left",
        validate="one_to_one",
    )
)

# Normalise comments only for evidence discovery.
selected_distance_group_structure["normalised_comment"] = (
    selected_distance_group_structure["selected_raw_comment"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Use a noncapturing expression to avoid the pandas regex warning seen
# previously. This flags explicit tie or amended-result wording only.
tie_or_amendment_pattern = (
    r"(?:dead[- ]?heat|deadheated|dead-heated|"
    r"disqualif|demot|promot|awarded|placed\s+\d|"
    r"finished\s+\d|outright)"
)

selected_distance_group_structure["comment_evidence_state"] = (
    "populated_without_explicit_tie_or_amendment"
)

selected_distance_group_structure.loc[
    selected_distance_group_structure[
        "normalised_comment"
    ].eq(""),
    "comment_evidence_state",
] = "blank_comment"

selected_distance_group_structure.loc[
    selected_distance_group_structure[
        "normalised_comment"
    ].str.contains(
        tie_or_amendment_pattern,
        regex=True,
        na=False,
    ),
    "comment_evidence_state",
] = "explicit_tie_or_amendment_evidence"

# Validate that each exceptional runner appears exactly once.
if len(selected_distance_group_structure) != 48:
    raise ValueError(
        "Expected 48 runner-level distance-group classifications, "
        f"but found {len(selected_distance_group_structure)}."
    )

if selected_distance_group_structure[
    "selected_source_rowid"
].duplicated().any():
    raise ValueError(
        "Each selected runner must have exactly one structural classification."
    )

# Summarise structure against available source-comment evidence.
distance_group_structure_summary = (
    selected_distance_group_structure
    .groupby(
        [
            "distance_group_structure",
            "comment_evidence_state",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        runner_rows=("selected_source_rowid", "size"),
        provisional_races=(
            "date",
            lambda values: len(
                set(
                    zip(
                        values,
                        selected_distance_group_structure.loc[
                            values.index,
                            "course",
                        ],
                        selected_distance_group_structure.loc[
                            values.index,
                            "off",
                        ],
                    )
                )
            ),
        ),
    )
    .sort_values(
        [
            "distance_group_structure",
            "comment_evidence_state",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

display(distance_group_structure_summary)

# Display only structurally separated pairs without explicit source-comment
# evidence. These are the smallest residual population likely to need closer
# review or external verification.
display(
    selected_distance_group_structure.loc[
        selected_distance_group_structure[
            "distance_group_structure"
        ].eq("separated_same_distance_pair")
        & ~selected_distance_group_structure[
            "comment_evidence_state"
        ].eq("explicit_tie_or_amendment_evidence"),
        [
            "date",
            "course",
            "off",
            "selected_horse",
            "selected_raw_pos",
            "numeric_ovr_btn",
            "counterpart_horses",
            "counterpart_positions",
            "selected_raw_comment",
        ],
    ]
    .sort_values(
        ["date", "course", "off"],
        kind="stable",
    )
    .reset_index(drop=True)
)

,distance_group_structure,comment_evidence_state,runner_rows,provisional_races
0,adjacent_same_distance_pair,blank_comment,11,11
1,adjacent_same_distance_pair,explicit_tie_or_amendment_evidence,8,8
2,adjacent_same_distance_pair,populated_without_explicit_tie_or_amendment,8,8
3,includes_nonnumeric_counterpart,explicit_tie_or_amendment_evidence,4,4
4,multi_runner_distance_plateau,blank_comment,9,3
5,multi_runner_distance_plateau,explicit_tie_or_amendment_evidence,1,1
6,multi_runner_distance_plateau,populated_without_explicit_tie_or_amendment,3,2
7,separated_same_distance_pair,blank_comment,1,1
8,separated_same_distance_pair,explicit_tie_or_amendment_evidence,3,3


,date,course,off,selected_horse,selected_raw_pos,numeric_ovr_btn,counterpart_horses,counterpart_positions,selected_raw_comment
0,2015-05-16,Morphettville (AUS),4:38,Mighty Maher (AUS),12,8.75,Cinnamon Carter (AUS),10,


### Residual separated same-distance pair

Only one of the 48 unique-position zero-margin rows remains without explicit tie or amendment evidence:

* Morphettville, 16 May 2015;
* Mighty Maher, stored position 12;
* Cinnamon Carter, stored position 10;
* both stored at `ovr_btn = 8.75`.

The race’s supplied position sequence is itself irregular:

* position 10 occurs twice;
* positions 8 and 13 are absent;
* the position-11 runner is stored at a different overall distance.

The available source and external evidence do not establish whether Mighty Maher physically tied with Cinnamon Carter, whether later amendments disrupted the position sequence, or whether the supplied position or distance values are defective.

This race should therefore remain an unresolved source-structure exception rather than being forced into the amended-tie interpretation.


## Conclusion

The source fields `ovr_btn` and `btn` both contain beaten-distance information, but they operate at different levels.

### `ovr_btn`

`ovr_btn` records the runner’s cumulative distance from the physical first-place reference used by the source result.

For ordinary results:

* the physical winner carries `ovr_btn = 0`;
* later finishers carry positive cumulative distances; and
* nonfinishers use the text sentinel `-`.

The field does not always align with the final official placing stored in `pos`.

Where a result is amended after the finish:

* `pos` may record the official revised order;
* `ovr_btn` may continue to reflect the physical finishing order; and
* the official winner may therefore carry a positive `ovr_btn`.

A later-positioned runner carrying `ovr_btn = 0` is highly diagnostic of a physical winner or dead-heat participant subsequently demoted, disqualified or separated by an amended result. It is not infallible because a small number of verified source defects also produce this structure.

### `btn`

`btn` records the margin from the preceding physical finisher or preceding stored distance group rather than the cumulative distance from the winner.

For ordinary sequential finishers:

* `btn` is positive;
* cumulative `ovr_btn` generally increases through the finishing order; and
* `btn` represents the incremental separation contributing to that cumulative distance.

A numeric `btn = 0` indicates that the runner has no recorded separation from at least one other runner carrying the same stored `ovr_btn`.

This commonly represents:

* an official dead heat;
* a physical dead heat later separated by an amended result;
* a multi-runner same-distance plateau; or
* grouped, rounded or capped distance presentation.

It does not, by itself, prove that the final official result contains a dead heat.

### Nonnumeric values

The only text value observed in either distance field is `-`.

This sentinel is associated with nonfinishers and should be parsed as unavailable beaten distance rather than as zero.

It must not be converted to a numeric margin.

### Recommended processed representation

The processed database should preserve the raw source fields and derive separate analytical values.

Recommended fields include:

* `raw_ovr_btn`;
* `raw_btn`;
* `overall_beaten_distance`;
* `previous_runner_margin`;
* `distance_available`;
* `distance_reference_stage`;
* `distance_value_origin`;
* `distance_verification_id`; and
* `distance_quality_status`.

The default processed interpretation should be:

* numeric `ovr_btn` → cumulative physical-finish distance;
* numeric `btn` → incremental physical-finish margin;
* text `-` → unavailable distance;
* positive `ovr_btn` on official position 1 → possible amended result or source defect;
* later-positioned `ovr_btn = 0` → possible demoted physical winner, physical dead heat or source defect;
* positive `ovr_btn` with `btn = 0` → membership of a same-distance group.

Raw source values must remain unchanged. Verified contradictions should be handled through explicit downstream corrections or supplements with provenance.

## Limitations

The investigation establishes source semantics from observed structure, source comments and governed external verification. It does not reconstruct every jurisdiction’s official margin-calculation rules.

Important limitations are:

1. **Physical versus official result stage**

   The distance fields may describe the order at the finish while `pos` describes a later amended official result. Downstream users must not assume that all three fields refer to the same result stage.

2. **Rounding and grouped margins**

   Equal `ovr_btn` values may represent exact ties, rounded values, capped distances or source grouping. The available data does not always distinguish these mechanisms.

3. **Source defects**

   A small number of externally verified races contain incorrect zero distances, omitted runners or inconsistent position sequences.

4. **Jurisdictional variation**

   Distance conventions vary between racing jurisdictions and source providers. The notebook identifies the dominant semantics of this dataset but does not claim universal racing-result rules.

5. **Manual verification coverage**

   External verification was targeted at bounded exceptional populations rather than every race in the source.

6. **No automatic correction from structure alone**

   Diagnostic patterns are strong enough for validation and review queues, but not for silently rewriting individual results without supporting source comments or governed external evidence.

## Final field decision

| Field                                   | Processed interpretation                                                         | Decision                  |
| --------------------------------------- | -------------------------------------------------------------------------------- | ------------------------- |
| `ovr_btn`                               | Cumulative distance from the source’s physical first-place reference             | Confirmed                 |
| `btn`                                   | Incremental margin from the preceding physical finisher or stored distance group | Confirmed                 |
| Text `-`                                | Beaten distance unavailable for a nonfinisher                                    | Confirmed                 |
| Official winner with positive `ovr_btn` | Amended-result indicator or source anomaly requiring review                      | Governed exception        |
| Later runner with `ovr_btn = 0`         | Demoted physical winner, physical dead heat or source defect                     | Governed exception        |
| Positive `ovr_btn` with `btn = 0`       | Same-distance group membership; not automatically an official dead heat          | Confirmed with limitation |
| Raw source alteration                   | Do not alter the immutable source                                                | Prohibited                |
| Verified corrections                    | Apply only through governed downstream reconciliation with provenance            | Required                  |
